# Feature Engineering
## PEAD (Post-Earnings Announcement Drift) Baseline Study
**Universe:** NVDA, AMD, MSFT, META, GOOGL, AMZN, TSLA, NKE, MCD, BKNG, NFLX, SPOT, DIS, JPM, GS, UNH, LLY, CAT, XOM, CRM  
**Period:** 2018-01-01 to 2024-12-31  
**Signal:** EPS Surprise (z-scored per ticker) vs Forward Log Returns

In [1]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# PEAD IC STUDY â€” 100 Random S&P 400 (Mid-Cap) Universe
# Spearman Information Coefficient: EPS Surprise z-score vs Forward Returns
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
import os, warnings, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas_market_calendars as mcal
from datetime import timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)

OUTPUT_DIR = r'C:\Trading Algo\stocks_sentiment_PEAD\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# â”€â”€ 100 random S&P 400 mid-cap stocks (Wikipedia source, seed=42)
TICKERS = [
    'ACI',  'ALK',  'ALLY', 'AM',   'AMG',  'APPF', 'AR',   'ASH',  'AVT',  'AXTA',
    'AYI',  'BCO',  'BIO',  'BLD',  'BRKR', 'BRX',  'BSY',  'CACI', 'CCK',  'CHE',
    'CNM',  'CNX',  'COLB', 'CRBG', 'CYTK', 'DCI',  'DOCU', 'DUOL', 'DY',   'EEFT',
    'EHC',  'ELF',  'ELS',  'ENSG', 'EVR',  'FFIN', 'FHI',  'FHN',  'FIVE', 'FLR',
    'FN',   'FTI',  'GAP',  'GT',   'HAE',  'HLNE', 'HOMB', 'HR',   'IDA',  'ILMN',
    'IPGP', 'IRT',  'JEF',  'KBR',  'KD',   'LECO', 'MANH', 'MEDP', 'MTSI', 'MUSA',
    'NEU',  'NFG',  'ONB',  'PEN',  'PFGC', 'PNFP', 'PPC',  'PSN',  'QLYS', 'RH',
    'RPM',  'RRC',  'SAIC', 'SAM',  'SARO', 'SFM',  'SITM', 'SLGN', 'SN',   'SR',
    'TEX',  'TKR',  'TTC',  'TXNM', 'UBSI', 'USFD', 'UTHR', 'VC',   'VICR', 'VOYA',
    'WAL',  'WCC',  'WFRD', 'WH',   'WLK',  'WMG',  'WPC',  'WTS',  'XPO',  'ZION',
]
START_DATE = '2018-01-01'
END_DATE   = '2026-06-20'
HORIZONS   = [5, 10, 21, 42, 63]

# â”€â”€ NYSE calendar for business day arithmetic â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
nyse      = mcal.get_calendar('NYSE')
schedule  = nyse.schedule(start_date='2017-01-01', end_date='2027-12-31')
tdays_idx = pd.DatetimeIndex(schedule.index).normalize()

def next_trading_day(dt):
    dt = pd.Timestamp(dt).normalize()
    i  = tdays_idx.searchsorted(dt)
    return tdays_idx[i] if i < len(tdays_idx) else None

def trading_day_offset(dt, n):
    dt = pd.Timestamp(dt).normalize()
    i  = tdays_idx.searchsorted(dt)
    j  = i + n
    return tdays_idx[j] if j < len(tdays_idx) else None

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 1 â€” Build earnings event table (parallel fetch)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('='*65)
print('STEP 1 â€” Building earnings event table')
print('='*65)

def _fetch_ic_earnings(ticker, retries=3):
    """Fetch earnings events for one ticker; return (rows, drop_counts, status)."""
    ticker_rows = []
    drops = dict(outside_range=0, bad_eps=0)
    for attempt in range(retries + 1):
        try:
            t  = yf.Ticker(ticker)
            ed = t.get_earnings_dates(limit=60)
            if ed is None or ed.empty:
                return ticker_rows, drops, 'no_data'
            ed = ed.copy()
            idx = pd.to_datetime(ed.index)
            ed.index = idx.tz_convert(None) if idx.tz is not None else idx
            col_map = {}
            for c in ed.columns:
                cl = c.lower().replace(' ', '_')
                if 'reported' in cl or 'actual' in cl: col_map[c] = 'eps_actual'
                elif 'estimate' in cl:                 col_map[c] = 'eps_estimate'
                elif 'surprise' in cl and '%' in cl:   col_map[c] = 'surprise_pct_raw'
            ed = ed.rename(columns=col_map)
            if 'eps_actual' not in ed.columns or 'eps_estimate' not in ed.columns:
                return ticker_rows, drops, 'missing_eps_cols'
            for date, row in ed.iterrows():
                if date < pd.Timestamp(START_DATE) or date > pd.Timestamp(END_DATE):
                    drops['outside_range'] += 1; continue
                ea = row.get('eps_actual',  np.nan)
                ee = row.get('eps_estimate', np.nan)
                if pd.isna(ee) or ee == 0:
                    drops['bad_eps'] += 1; continue
                if pd.isna(ea):
                    drops['bad_eps'] += 1; continue
                eps_surprise_pct = (float(ea) - float(ee)) / abs(float(ee)) * 100
                trade_start = next_trading_day(date + timedelta(days=1))
                ticker_rows.append(dict(
                    ticker=ticker, earnings_date=date,
                    eps_actual=float(ea), eps_estimate=float(ee),
                    eps_surprise_pct=eps_surprise_pct,
                    announcement_time='AMC',
                    trade_start_date=trade_start,
                ))
            return ticker_rows, drops, 'ok'
        except Exception as ex:
            if attempt < retries:
                time.sleep(15)
            else:
                return ticker_rows, drops, f'error: {ex}'
    return ticker_rows, drops, 'error: max retries exceeded'

rows     = []
drop_log = dict(outside_range=0, bad_eps=0, api_no_data=0)

print(f'Fetching earnings for {len(TICKERS)} tickers (2 parallel workers, retry on rate-limit)...')
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(_fetch_ic_earnings, t): t for t in TICKERS}
    done = 0
    for future in as_completed(futures):
        ticker = futures[future]
        t_rows, t_drops, status = future.result()
        rows.extend(t_rows)
        drop_log['outside_range'] += t_drops['outside_range']
        drop_log['bad_eps']       += t_drops['bad_eps']
        done += 1
        if status != 'ok':
            print(f'  {ticker}: {status}')
            drop_log['api_no_data'] += 1
        if done % 25 == 0:
            print(f'  {done}/{len(TICKERS)} done â€” {len(rows)} events so far')

events = pd.DataFrame(rows)
if events.empty:
    raise RuntimeError(
        "No earnings data fetched for any ticker â€” yfinance is rate-limiting all requests. "
        "Wait 10-15 minutes and re-run."
    )

print(f'\nData quality notes:')
print(f'  Rows outside 2018-2026 date range:      {drop_log["outside_range"]}')
print(f'  Rows dropped (eps_estimate zero/NaN):   {drop_log["bad_eps"]}')
print(f'  Tickers with no API data:               {drop_log["api_no_data"]}')
print(f'\nEvents table shape: {events.shape}')
print(events.head(10).to_string())

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 2 â€” Normalize EPS surprise (z-score per ticker)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('STEP 2 â€” Z-scoring EPS surprise per ticker')
print('=' * 65)

def zscore_per_ticker(df):
    df = df.copy()
    def _z(g):
        mu, sigma = g['eps_surprise_pct'].mean(), g['eps_surprise_pct'].std()
        g = g.copy()
        g['eps_surprise_zscore'] = (g['eps_surprise_pct'] - mu) / sigma if sigma > 0 else 0.0
        return g
    return df.groupby('ticker', group_keys=False).apply(_z)

events = zscore_per_ticker(events)
print('Z-score stats per ticker:')
print(events.groupby('ticker')['eps_surprise_zscore'].agg(['mean','std','count']).to_string())

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 3 â€” Compute forward log returns
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('STEP 3 â€” Computing forward log returns')
print('=' * 65)

price_data = yf.download(
    TICKERS,
    start='2017-12-01',
    end='2026-06-21',
    auto_adjust=True,
    progress=False
)['Close']
price_data.index = pd.to_datetime(price_data.index).normalize()

drop_fwd = 0
fwd_cols = {h: [] for h in HORIZONS}

for idx, row in events.iterrows():
    ticker = row['ticker']
    t0     = row['trade_start_date']
    if t0 is None or ticker not in price_data.columns:
        for h in HORIZONS: fwd_cols[h].append(np.nan)
        drop_fwd += 1; continue

    try:
        p0 = float(price_data.loc[t0, ticker])
    except KeyError:
        avail = price_data[ticker].dropna()
        avail = avail[avail.index >= t0]
        p0 = float(avail.iloc[0]) if len(avail) > 0 else np.nan

    if pd.isna(p0) or p0 <= 0:
        for h in HORIZONS: fwd_cols[h].append(np.nan)
        drop_fwd += 1; continue

    for h in HORIZONS:
        t1 = trading_day_offset(t0, h)
        if t1 is None:
            fwd_cols[h].append(np.nan); continue
        try:
            p1 = float(price_data.loc[t1, ticker])
        except KeyError:
            avail = price_data[ticker].dropna()
            avail = avail[avail.index >= t1]
            p1 = float(avail.iloc[0]) if len(avail) > 0 else np.nan
        fwd_cols[h].append(np.log(p1 / p0) if (not pd.isna(p1) and p1 > 0) else np.nan)

for h in HORIZONS:
    events[f'fwd_ret_{h}d'] = fwd_cols[h]

ret_cols = [f'fwd_ret_{h}d' for h in HORIZONS]
before = len(events)
events.dropna(subset=ret_cols, how='all', inplace=True)
after = len(events)
print(f'  Dropped {before - after} rows with no forward returns computable')
print(f'  Final events: {after}')
print(events[['ticker','earnings_date','eps_surprise_zscore'] + ret_cols].head(10).to_string())

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 4 â€” Spearman IC Analysis
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('STEP 4 â€” Spearman IC Analysis')
print('=' * 65)

def bootstrap_ci(x, y, n_boot=1000, ci=0.95, seed=42):
    rng  = np.random.default_rng(seed)
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    n    = len(x)
    ic_boot = [stats.spearmanr(x[rng.integers(0, n, n)], y[rng.integers(0, n, n)])[0]
               for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return np.quantile(ic_boot, alpha), np.quantile(ic_boot, 1 - alpha)

X = events['eps_surprise_zscore'].values
summary_rows = []

for h in HORIZONS:
    col  = f'fwd_ret_{h}d'
    Y    = events[col].values
    mask = ~(np.isnan(X) | np.isnan(Y))
    n    = mask.sum()
    ic, pval = stats.spearmanr(X[mask], Y[mask])
    t_stat   = ic * np.sqrt(n) / np.sqrt(max(1 - ic**2, 1e-12))
    ci_lo, ci_hi = bootstrap_ci(X, Y)
    summary_rows.append({
        'horizon': f'{h}d', 'N': n,
        'IC': round(ic,4), 't-stat': round(t_stat,3), 'p-value': round(pval,4),
        'CI_lower': round(ci_lo,4), 'CI_upper': round(ci_hi,4),
        'significant': pval < 0.05,
    })

ic_summary = pd.DataFrame(summary_rows)
print('\nPooled Spearman IC Summary (all tickers):')
print(ic_summary.to_string(index=False))

print('\nPer-ticker Spearman IC at 21-day horizon:')
ticker_ic = []
for tk in TICKERS:
    sub  = events[events['ticker'] == tk]
    xx   = sub['eps_surprise_zscore'].values
    yy   = sub['fwd_ret_21d'].values
    mask = ~(np.isnan(xx) | np.isnan(yy))
    if mask.sum() < 4:
        ticker_ic.append({'ticker': tk, 'IC_21d': np.nan, 'N': mask.sum()}); continue
    r, p = stats.spearmanr(xx[mask], yy[mask])
    ticker_ic.append({'ticker': tk, 'IC_21d': round(r,4), 'N': mask.sum(), 'p': round(p,4)})

ticker_ic_df = pd.DataFrame(ticker_ic)
print(ticker_ic_df.to_string(index=False))

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 5 â€” IC Decay Plot
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('STEP 5 â€” IC Decay Plot')
print('=' * 65)

fig, ax = plt.subplots(figsize=(9, 5))
horizon_vals = [5, 10, 21, 42, 63]
ic_vals  = ic_summary['IC'].values
ci_lo_v  = ic_summary['CI_lower'].values
ci_hi_v  = ic_summary['CI_upper'].values

ax.errorbar(horizon_vals, ic_vals,
            yerr=[np.maximum(0, ic_vals - ci_lo_v), np.maximum(0, ci_hi_v - ic_vals)],
            fmt='o-', color='steelblue', lw=2, markersize=8,
            capsize=5, capthick=1.5, ecolor='cornflowerblue', label='Spearman IC')
ax.axhline(0, color='black', linestyle='--', linewidth=1)
for i, (x, y) in enumerate(zip(horizon_vals, ic_vals)):
    sig = '*' if ic_summary.iloc[i]['significant'] else ''
    ax.annotate(f'{y:.3f}{sig}', xy=(x, y), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=9)

ax.set_xlabel('Horizon (calendar days)', fontsize=12)
ax.set_ylabel('Spearman IC', fontsize=12)
ax.set_title('PEAD Spearman IC Decay â€” EPS Surprise vs Forward Returns\n(100-stock S&P 400 mid-cap universe)',
             fontsize=13, fontweight='bold')
ax.set_xticks(horizon_vals); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'pead_ic_decay.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_ic_decay.png')

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 6 â€” Per-Ticker IC Heatmap (100 tickers Ã— 5 horizons)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('STEP 6 â€” Per-Ticker IC Heatmap')
print('=' * 65)

heatmap_rows = []
for tk in TICKERS:
    sub    = events[events['ticker'] == tk]
    row_ic = {'ticker': tk}
    for h in HORIZONS:
        col  = f'fwd_ret_{h}d'
        xx   = sub['eps_surprise_zscore'].values
        yy   = sub[col].values
        mask = ~(np.isnan(xx) | np.isnan(yy))
        if mask.sum() < 4:
            row_ic[f'{h}d'] = np.nan
        else:
            r, _ = stats.spearmanr(xx[mask], yy[mask])
            row_ic[f'{h}d'] = round(r, 4)
    heatmap_rows.append(row_ic)

heatmap_df = pd.DataFrame(heatmap_rows).set_index('ticker')
heatmap_df.columns = [f'{h}d' for h in HORIZONS]

fig, ax = plt.subplots(figsize=(10, 30))
vals = heatmap_df.values[~np.isnan(heatmap_df.values)]
vmax = max(abs(vals).max(), 0.01) if len(vals) > 0 else 0.5
sns.heatmap(
    heatmap_df, ax=ax,
    cmap='RdYlGn', center=0, vmin=-vmax, vmax=vmax,
    annot=True, fmt='.3f', linewidths=0.5,
    annot_kws={'fontsize': 5},
    cbar_kws={'label': 'Spearman IC'}
)
ax.set_title('Per-Ticker Spearman IC: EPS Surprise vs Forward Returns\n(100-stock S&P 400 mid-cap universe)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Horizon', fontsize=11)
ax.set_ylabel('Ticker', fontsize=11)
ax.tick_params(axis='y', labelsize=7)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'pead_ic_heatmap.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_ic_heatmap.png')

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 7 â€” Quintile Return Analysis
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('STEP 7 â€” Quintile Return Analysis (21-day horizon)')
print('=' * 65)

ev_21 = events[['eps_surprise_zscore', 'fwd_ret_21d']].dropna().copy()
ev_21['quintile'] = pd.qcut(ev_21['eps_surprise_zscore'], q=5,
                             labels=['Q1\n(Worst)','Q2','Q3','Q4','Q5\n(Best)'])

quint_stats = ev_21.groupby('quintile', observed=False)['fwd_ret_21d'].agg(
    mean='mean', median='median', count='count'
).reset_index()
quint_stats['mean_pct']   = (np.exp(quint_stats['mean'])   - 1) * 100
quint_stats['median_pct'] = (np.exp(quint_stats['median']) - 1) * 100
print(quint_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(5)
width = 0.35
colors_mean   = ['#d73027' if v < 0 else '#1a9850' for v in quint_stats['mean_pct']]
colors_median = ['#fc8d59' if v < 0 else '#91cf60' for v in quint_stats['median_pct']]

bars1 = ax.bar(x - width/2, quint_stats['mean_pct'],   width,
               label='Mean return (%)',   color=colors_mean,   alpha=0.85)
bars2 = ax.bar(x + width/2, quint_stats['median_pct'], width,
               label='Median return (%)', color=colors_median, alpha=0.85)

ax.axhline(0, color='black', linewidth=0.8)
for bar in bars1:
    h = bar.get_height()
    ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 4 if h >= 0 else -14), textcoords='offset points',
                ha='center', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 4 if h >= 0 else -14), textcoords='offset points',
                ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(quint_stats['quintile'])
ax.set_xlabel('EPS Surprise Quintile', fontsize=12)
ax.set_ylabel('21-Day Forward Return (%)', fontsize=12)
ax.set_title('PEAD Quintile Return Analysis â€” 21-Day Horizon\n(Q1 = most negative, Q5 = most positive | 100-stock universe)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'pead_quintile_returns.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_quintile_returns.png')

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# FINAL INTERPRETATION
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '=' * 65)
print('INTERPRETATION')
print('=' * 65)

ic_21_row    = ic_summary[ic_summary['horizon'] == '21d'].iloc[0]
ic_21        = ic_21_row['IC']
pval_21      = ic_21_row['p-value']
sig_21       = ic_21_row['significant']
q1_mean      = quint_stats[quint_stats['quintile'] == 'Q1\n(Worst)']['mean_pct'].values[0]
q5_mean      = quint_stats[quint_stats['quintile'] == 'Q5\n(Best)']['mean_pct'].values[0]
q5_q1_spread = q5_mean - q1_mean

means_list = quint_stats['mean_pct'].tolist()
monotonic  = all(means_list[i+1] >= means_list[i] for i in range(len(means_list)-1))

n_sig           = ic_summary['significant'].sum()
all_positive_ic = (ic_summary['IC'] > 0).all()

print(f"""
â”€â”€â”€ Plain-English Summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

STATISTICAL SIGNIFICANCE:
  â€¢ {n_sig} of 5 horizons show statistically significant IC (p < 0.05).
  â€¢ At the canonical 21-day horizon: IC = {ic_21:.4f}, p = {pval_21:.4f}.
    This is {'SIGNIFICANT' if sig_21 else 'NOT significant'} at the 5% level.
  â€¢ IC confidence interval (95%, bootstrap): [{ic_21_row['CI_lower']:.4f}, {ic_21_row['CI_upper']:.4f}].

ECONOMIC MAGNITUDE:
  â€¢ Q5 (most positive EPS surprise) mean 21-day return: {q5_mean:.2f}%
  â€¢ Q1 (most negative EPS surprise) mean 21-day return: {q1_mean:.2f}%
  â€¢ Q5 - Q1 spread: {q5_q1_spread:.2f}% over 21 calendar days.
  â€¢ Quintile relationship is {'MONOTONIC (consistent with PEAD)' if monotonic else 'NOT monotonic â€” uneven cross-quintile pattern'}.

OVERALL VERDICT:
""")

if sig_21 and abs(ic_21) >= 0.05 and abs(q5_q1_spread) >= 2.0:
    print("  âœ“ PEAD signal appears STATISTICALLY SIGNIFICANT and ECONOMICALLY")
    print("    MEANINGFUL in this 100-stock universe over 2018-2026.")
    print(f"    An IC of {ic_21:.4f} is modest but reliable at this horizon.")
    print(f"    The {q5_q1_spread:.1f}% Q5-Q1 spread suggests a tradeable edge,")
    print("    though transaction costs, market impact, and risk adjustment")
    print("    need to be verified before live implementation.")
elif sig_21 and abs(ic_21) < 0.05:
    print("  âš   Signal is statistically significant but ECONOMICALLY WEAK.")
    print(f"    IC = {ic_21:.4f} is small; effect may not survive transaction costs.")
elif not sig_21:
    print("  âœ— PEAD signal is NOT statistically significant at the 21-day horizon")
    print("    in this sample. Either the effect has decayed, or the EPS surprise")
    print("    estimate quality is limiting the signal.")
else:
    print("  âš   Mixed evidence â€” review IC decay plot and heatmap for details.")

print(f"""
CROSS-SECTIONAL NOTE:
  Per-ticker IC at 21d is heterogeneous â€” some tickers show strong
  positive PEAD, others near zero or negative. This suggests ticker-
  level segmentation (e.g., sector, market-cap weighting, estimate
  quality) may improve signal reliability in a production factor model.

Outputs saved to: {OUTPUT_DIR}
  â€¢ pead_ic_decay.png
  â€¢ pead_ic_heatmap.png
  â€¢ pead_quintile_returns.png
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
""")


STEP 1 â€” Building earnings event table
Fetching earnings for 100 tickers (2 parallel workers, retry on rate-limit)...


  25/100 done â€” 796 events so far


  50/100 done â€” 1625 events so far


  75/100 done â€” 2395 events so far


  100/100 done â€” 3184 events so far

Data quality notes:
  Rows outside 2018-2026 date range:      4139
  Rows dropped (eps_estimate zero/NaN):   16
  Tickers with no API data:               0

Events table shape: (3184, 7)
  ticker       earnings_date  eps_actual  eps_estimate  eps_surprise_pct announcement_time trade_start_date
0    ACI 2026-04-14 11:00:00        0.48          0.44          9.090909               AMC       2026-04-15
1    ACI 2026-01-07 12:00:00        0.72          0.68          5.882353               AMC       2026-01-08
2    ACI 2025-10-14 11:00:00        0.44          0.40         10.000000               AMC       2025-10-15
3    ACI 2025-07-15 11:00:00        0.55          0.54          1.851852               AMC       2025-07-16
4    ACI 2025-04-15 11:00:00        0.46          0.41         12.195122               AMC       2025-04-16
5    ACI 2025-01-08 12:00:00        0.71          0.64         10.937500               AMC       2025-01-10
6    ACI 2024-10-1

...

  Dropped 5 rows with no forward returns computable
  Final events: 3179
  ticker       earnings_date  eps_surprise_zscore  fwd_ret_5d  fwd_ret_10d  fwd_ret_21d  fwd_ret_42d  fwd_ret_63d
0    ACI 2026-04-14 11:00:00            -0.425974    0.019684     0.002879     0.004698    -0.100596          NaN
1    ACI 2026-01-07 12:00:00            -0.544441    0.044112     0.034298     0.096783    -0.002721     0.047111
2    ACI 2025-10-14 11:00:00            -0.392409    0.018056    -0.042331    -0.059445    -0.095153    -0.092272
3    ACI 2025-07-15 11:00:00            -0.693256    0.013066    -0.045683    -0.066966    -0.102207    -0.056527
4    ACI 2025-04-15 11:00:00            -0.311360    0.055634     0.046588     0.050669     0.006258    -0.025948
5    ACI 2025-01-08 12:00:00            -0.357794   -0.015661     0.000065     0.072621     0.067913     0.067913
6    ACI 2024-10-15 12:00:00            -0.530867    0.033829     0.002161     0.048780     0.087296     0.076660
7    ACI 2024-0


Pooled Spearman IC Summary (all tickers):
horizon    N     IC  t-stat  p-value  CI_lower  CI_upper  significant
     5d 3179 0.0016   0.093   0.9262   -0.0353    0.0367        False
    10d 3176 0.0008   0.045   0.9641   -0.0358    0.0354        False
    21d 3169 0.0157   0.885   0.3762   -0.0326    0.0330        False
    42d 3091 0.0367   2.043   0.0412   -0.0373    0.0342         True
    63d 3081 0.0321   1.782   0.0749   -0.0320    0.0336        False

Per-ticker Spearman IC at 21-day horizon:
ticker  IC_21d  N      p
   ACI -0.0443 24 0.8370
   ALK  0.0374 34 0.8335
  ALLY -0.0582 34 0.7437
    AM  0.0009 34 0.9959
   AMG  0.0500 34 0.7790
  APPF  0.0411 34 0.8175
    AR -0.1837 34 0.2984
   ASH -0.2278 34 0.1950
   AVT -0.1068 34 0.5477
  AXTA  0.0029 34 0.9870
   AYI  0.3561 34 0.0387
   BCO  0.1254 34 0.4796
   BIO -0.3843 34 0.0248
   BLD  0.1386 34 0.4344
  BRKR  0.2330 34 0.1847
   BRX  0.1974 34 0.2631
   BSY  0.6030 23 0.0023
  CACI -0.2571 34 0.1421
   CCK -0.1512 34 0

  Saved: pead_ic_decay.png

STEP 6 â€” Per-Ticker IC Heatmap


  Saved: pead_ic_heatmap.png

STEP 7 â€” Quintile Return Analysis (21-day horizon)
   quintile     mean   median  count  mean_pct  median_pct
Q1\n(Worst) 0.007216 0.016377    634  0.724237    1.651216
         Q2 0.001632 0.008042    634  0.163340    0.807488
         Q3 0.004234 0.009338    633  0.424288    0.938146
         Q4 0.012371 0.013264    635  1.244766    1.335219
 Q5\n(Best) 0.014092 0.018780    633  1.419155    1.895731


  Saved: pead_quintile_returns.png

INTERPRETATION

â”€â”€â”€ Plain-English Summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

STATISTICAL SIGNIFICANCE:
  â€¢ 1 of 5 horizons show statistically significant IC (p < 0.05).
  â€¢ At the canonical 21-day horizon: IC = 0.0157, p = 0.3762.
    This is NOT significant at the 5% level.
  â€¢ IC confidence interval (95%, bootstrap): [-0.0326, 0.0330].

ECONOMIC MAGNITUDE:
  â€¢ Q5 (most positive EPS surprise) mean 21-day return: 1.42%
  â€¢ Q1 (most negative EPS surprise) mean 21-day return: 0.72%
  â€¢ Q5 - Q1 spread: 0.69% over 21 calendar days.
  â€¢ Quintile relationship is NOT monotonic â€” uneven cross-quintile pattern.

OVERALL VERDICT:

  âœ— PEAD signal is NOT statistically significant at the 21-day horizon
    in this sample. Either the effect has decayed, or the EPS surprise
    estimate quality is limiting the signal.

CROSS-SECTIONAL NOTE:
  Per-ticker I

In [2]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# PEAD LONG-ONLY BACKTEST â€” Full S&P 500 Universe (499 stocks)
# Hypothesis: Post-Earnings Institutional Accumulation (PEIA)
# Signal: gap>0 + price holds above gap open + volume â‰¥1.2Ã— pre-earnings baseline
# Confirmation window: 3 trading days | Entry: day+4 open | Hold: 63 calendar days
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
import os, warnings, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from datetime import timedelta
import pandas_market_calendars as mcal

warnings.filterwarnings('ignore')
np.random.seed(42)

OUTPUT_DIR = r'C:\Trading Algo\stocks_sentiment_PEAD\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# â”€â”€ Full S&P 500 (499 stocks, BRK/GOOG/FOX/NWS class-share duplicates removed)
TICKERS = [
    'MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A',
    'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL',
    'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP',
    'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP',
    'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK',
    'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX',
    'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BNY', 'BA', 'BKNG', 'BSX',
    'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS',
    'CPT', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CASY', 'CAT', 'CBOE', 'CBRE',
    'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG',
    'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX',
    'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'CAG',
    'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP',
    'COST', 'CRH', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG',
    'DVA', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG',
    'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD',
    'ETN', 'EBAY', 'SATS', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EME', 'EMR',
    'ETR', 'EOG', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG',
    'EVRG', 'ES', 'EXC', 'EXE', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS',
    'FICO', 'FAST', 'FRT', 'FDX', 'FDXF', 'FIS', 'FITB', 'FSLR', 'FE', 'FISV',
    'FLEX', 'F', 'FTNT', 'FTV', 'FOXA', 'BEN', 'FCX', 'GRMN', 'IT', 'GE',
    'GEHC', 'GEV', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN',
    'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY',
    'HPE', 'HLT', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM',
    'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC',
    'IBKR', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM',
    'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KVUE', 'KDP', 'KEY',
    'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH',
    'LRCX', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LMT', 'L',
    'LOW', 'LULU', 'LITE', 'LYB', 'MTB', 'MPC', 'MAR', 'MRSH', 'MLM', 'MRVL',
    'MAS', 'MA', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD',
    'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'TAP', 'MDLZ', 'MPWR', 'MNST',
    'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA',
    'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE',
    'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL',
    'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PSKY', 'PH', 'PAYX', 'PYPL', 'PNR',
    'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'PPG', 'PPL', 'PFG',
    'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM',
    'DGX', 'Q', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG',
    'RMD', 'RVTY', 'HOOD', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM',
    'SNDK', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM',
    'SW', 'SNA', 'SOLV', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE',
    'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP',
    'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX',
    'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN',
    'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS',
    'VLO', 'VEEV', 'VTR', 'VLTO', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VRT', 'VTRS',
    'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD',
    'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB',
    'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS',
]
START_DATE     = '2018-01-01'
END_DATE       = '2026-06-20'
LONG_HOLD_DAYS = 63
INIT_CAP       = 100_000.0
# POS_SIZE and MAX_SLOTS are derived dynamically in STEP 3 from peak overlap.
TC_BPS         = 5
PRIMARY_Z      = 0.5
THRESHOLDS     = [0.0, 0.25, 0.5, 0.75, 1.0]
RF_ANNUAL      = 0.04
CONFIRM_DAYS     = 3    # trading days to observe post-earnings (days 1-3)
VOL_RATIO_MIN    = 1.2  # post-earnings avg volume must be â‰¥ this Ã— pre-earnings baseline
PRE_EARN_VOL_DAYS = 20  # trading days before earnings for volume baseline

# â”€â”€ Trading calendar â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
nyse      = mcal.get_calendar('NYSE')
schedule  = nyse.schedule(start_date='2017-01-01', end_date='2027-12-31')
tdays_idx = pd.DatetimeIndex(schedule.index).normalize()

def next_tday(dt):
    dt = pd.Timestamp(dt).normalize()
    i  = tdays_idx.searchsorted(dt)
    return tdays_idx[i] if i < len(tdays_idx) else None

def tday_offset(dt, n_cal):
    target = pd.Timestamp(dt).normalize() + timedelta(days=n_cal)
    i = tdays_idx.searchsorted(target)
    return tdays_idx[i] if i < len(tdays_idx) else None

def tday_offset_td(dt, n_td):
    dt = pd.Timestamp(dt).normalize()
    i  = int(tdays_idx.searchsorted(dt))
    j  = i + n_td
    return tdays_idx[j] if j < len(tdays_idx) else None

def prev_tday(dt):
    dt = pd.Timestamp(dt).normalize()
    i  = tdays_idx.searchsorted(dt)
    return tdays_idx[i - 1] if i > 0 else None

def scalar(v):
    if isinstance(v, (pd.Series, pd.DataFrame)):
        return float(v.iloc[0]) if len(v) > 0 else np.nan
    return float(v)

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 1 â€” Rebuild earnings event table (parallel fetch)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('='*65)
print('STEP 1 â€” Building earnings event table')
print('='*65)

print('Downloading price data...')
raw_px = yf.download(
    TICKERS, start='2017-12-01', end='2026-06-21',
    auto_adjust=True, progress=False
)
open_px  = raw_px['Open'].copy()
close_px = raw_px['Close'].copy()
vol_px   = raw_px['Volume'].copy()
open_px.index  = pd.to_datetime(open_px.index).normalize()
close_px.index = pd.to_datetime(close_px.index).normalize()
vol_px.index   = pd.to_datetime(vol_px.index).normalize()
print(f'  Price data shape: {close_px.shape}')

def _fetch_earnings(ticker):
    """Fetch and pre-process earnings events for one ticker.
    Returns (rows_list, drop_counts_dict, status_str).
    """
    ticker_rows = []
    drops = dict(outside_range=0, bad_eps=0, no_price=0)
    try:
        t  = yf.Ticker(ticker)
        ed = t.get_earnings_dates(limit=60)
        if ed is None or ed.empty:
            return ticker_rows, drops, 'no_data'

        ed = ed.copy()
        idx = pd.to_datetime(ed.index)
        ed.index = idx.tz_convert(None) if idx.tz is not None else idx
        col_map = {}
        for c in ed.columns:
            cl = c.lower().replace(' ', '_')
            if 'reported' in cl or 'actual' in cl: col_map[c] = 'eps_actual'
            elif 'estimate' in cl:                 col_map[c] = 'eps_estimate'
        ed = ed.rename(columns=col_map)

        if 'eps_actual' not in ed.columns or 'eps_estimate' not in ed.columns:
            return ticker_rows, drops, 'missing_eps_cols'

        for date, row in ed.iterrows():
            if date < pd.Timestamp(START_DATE) or date > pd.Timestamp(END_DATE):
                drops['outside_range'] += 1; continue
            ea = row.get('eps_actual',  np.nan)
            ee = row.get('eps_estimate', np.nan)
            if pd.isna(ee) or ee == 0 or pd.isna(ea):
                drops['bad_eps'] += 1; continue

            surp_pct    = (float(ea) - float(ee)) / abs(float(ee)) * 100
            trade_start = next_tday(date + timedelta(days=1))
            if trade_start is None: drops['no_price'] += 1; continue
            prev_day = prev_tday(trade_start)
            if prev_day is None:    drops['no_price'] += 1; continue

            try:
                p_open = scalar(open_px.loc[trade_start, ticker])
                p_prev = scalar(close_px.loc[prev_day,   ticker])
                if np.isnan(p_open) or np.isnan(p_prev) or p_prev <= 0:
                    drops['no_price'] += 1; continue
                gap_pct = (p_open - p_prev) / p_prev * 100
            except (KeyError, IndexError):
                drops['no_price'] += 1; continue

            ticker_rows.append(dict(
                ticker=ticker, earnings_date=date,
                eps_actual=float(ea), eps_estimate=float(ee),
                eps_surprise_pct=surp_pct,
                announcement_time='AMC',
                trade_start_date=trade_start,
                gap_pct=gap_pct,
            ))
        return ticker_rows, drops, 'ok'
    except Exception as ex:
        return ticker_rows, drops, f'error: {ex}'

def _fetch_earnings_retry(ticker, retries=2):
    for attempt in range(retries + 1):
        rows_r, drops_r, status_r = _fetch_earnings(ticker)
        if not status_r.startswith('error'):
            return rows_r, drops_r, status_r
        if attempt < retries:
            time.sleep(15)
    return rows_r, drops_r, status_r

rows     = []
drop_log = dict(outside_range=0, bad_eps=0, no_price=0, api_error=0)

print(f'Fetching earnings for {len(TICKERS)} tickers (3 parallel workers, retry on error)...')
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(_fetch_earnings_retry, t): t for t in TICKERS}
    done = 0
    for future in as_completed(futures):
        ticker = futures[future]
        t_rows, t_drops, status = future.result()
        rows.extend(t_rows)
        for k in ('outside_range', 'bad_eps', 'no_price'):
            drop_log[k] += t_drops[k]
        done += 1
        if status != 'ok':
            print(f'  {ticker}: {status}')
            drop_log['api_error'] += 1
        if done % 50 == 0:
            print(f'  {done}/{len(TICKERS)} tickers done â€” {len(rows)} events so far')

events_raw = pd.DataFrame(rows)
if events_raw.empty:
    raise RuntimeError(
        "No earnings data fetched â€” yfinance is rate-limiting all requests. "
        "Wait 10-15 minutes and re-run."
    )
total_seen = sum(drop_log.values()) + len(events_raw)
print(f'\nTotal events seen            : {total_seen}')
print(f'  Outside 2018-2026 / future  : {drop_log["outside_range"]}')
print(f'  Bad EPS (zero/NaN)          : {drop_log["bad_eps"]}')
print(f'  No price data               : {drop_log["no_price"]}')
print(f'  API error / no data         : {drop_log["api_error"]}')
print(f'Events after cleaning        : {len(events_raw)}')

def zscore_group(df):
    def _z(g):
        mu, sd = g['eps_surprise_pct'].mean(), g['eps_surprise_pct'].std()
        g = g.copy()
        g['eps_surprise_zscore'] = (g['eps_surprise_pct'] - mu) / sd if sd > 0 else 0.0
        return g
    return df.groupby('ticker', group_keys=False).apply(_z)

events_raw = zscore_group(events_raw)
print(events_raw[['ticker','earnings_date','eps_surprise_pct','eps_surprise_zscore','gap_pct']].head(10).to_string())

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 2 â€” Signal classification (PEIA: price hold + volume confirmation)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n'+'='*65)
print('STEP 2 â€” Entry signal classification (PEIA)')
print('='*65)

def classify_signals(ev, threshold):
    """
    Primary  : z > threshold AND gap > 0
    Confirmation (observed over CONFIRM_DAYS trading days post-earnings):
      1. Price close on day+CONFIRM_DAYS >= gap open on day+1  (gap held)
      2. Avg daily volume days 1..CONFIRM_DAYS >= VOL_RATIO_MIN Ã— 20-day pre-earnings avg
    Entry: open on day+(CONFIRM_DAYS+1) â€” fully look-ahead free.
    """
    ev = ev.copy()
    z, g = ev['eps_surprise_zscore'], ev['gap_pct']
    ev['signal']     = 'NONE'
    ev['vol_ratio']  = np.nan
    ev['entry_date'] = ev['trade_start_date']

    primary_mask = (z > threshold) & (g > 0)
    n_primary = primary_mask.sum()
    n_confirmed = n_price_fail = n_vol_fail = n_nodata = 0

    for idx in ev[primary_mask].index:
        row         = ev.loc[idx]
        ticker      = row['ticker']
        trade_start = row['trade_start_date']   # day+1

        # Observation close: day+CONFIRM_DAYS (e.g. day+3)
        confirm_close_day = tday_offset_td(trade_start, CONFIRM_DAYS - 1)
        # Entry open: day+CONFIRM_DAYS+1 (e.g. day+4)
        entry_day = tday_offset_td(trade_start, CONFIRM_DAYS)

        if confirm_close_day is None or entry_day is None:
            n_nodata += 1; continue

        # â”€â”€ Price confirmation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        try:
            gap_open      = scalar(open_px.loc[trade_start,       ticker])
            close_confirm = scalar(close_px.loc[confirm_close_day, ticker])
        except (KeyError, IndexError):
            n_nodata += 1; continue

        if np.isnan(gap_open) or np.isnan(close_confirm) or gap_open <= 0:
            n_nodata += 1; continue

        price_ok = (close_confirm >= gap_open)

        # â”€â”€ Volume confirmation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        ts_idx = int(tdays_idx.searchsorted(trade_start))
        if ts_idx < PRE_EARN_VOL_DAYS:
            n_nodata += 1; continue

        pre_dates  = tdays_idx[ts_idx - PRE_EARN_VOL_DAYS : ts_idx]
        post_dates = [tday_offset_td(trade_start, k) for k in range(CONFIRM_DAYS)]
        post_dates = [d for d in post_dates if d is not None]

        try:
            pre_vol  = [scalar(vol_px.loc[d, ticker]) for d in pre_dates  if d in vol_px.index]
            post_vol = [scalar(vol_px.loc[d, ticker]) for d in post_dates if d in vol_px.index]
        except (KeyError, IndexError):
            n_nodata += 1; continue

        pre_vol  = [v for v in pre_vol  if not np.isnan(v) and v > 0]
        post_vol = [v for v in post_vol if not np.isnan(v) and v > 0]

        if len(pre_vol) < 10 or len(post_vol) < 2:
            n_nodata += 1; continue

        vol_ratio = np.mean(post_vol) / np.mean(pre_vol)
        ev.at[idx, 'vol_ratio'] = vol_ratio
        vol_ok = (vol_ratio >= VOL_RATIO_MIN)

        if price_ok and vol_ok:
            ev.at[idx, 'signal']     = 'LONG'
            ev.at[idx, 'entry_date'] = entry_day
            n_confirmed += 1
        elif not price_ok:
            n_price_fail += 1
        else:
            n_vol_fail += 1

    print(f'  z>{threshold:.2f}+gap>0 primary signals   : {n_primary}')
    print(f'    Price+volume confirmed         : {n_confirmed}')
    print(f'    Failed â€” gap faded             : {n_price_fail}')
    print(f'    Failed â€” volume too low        : {n_vol_fail}')
    print(f'    No price/vol data              : {n_nodata}')
    return ev

print('Signal counts at each threshold (PEIA filter ON):')
for thr in THRESHOLDS:
    ev_t = classify_signals(events_raw, thr)
    n_l  = (ev_t['signal'] == 'LONG').sum()
    print(f'  z>{thr:.2f}: {n_l} LONG signals after PEIA filter')

print(f'\n--- Primary threshold = {PRIMARY_Z} (full detail) ---')
events = classify_signals(events_raw, PRIMARY_Z)
events = events[events['signal'] == 'LONG'].copy().reset_index(drop=True)
print(f'Final LONG signals entering backtest : {len(events)} @ {LONG_HOLD_DAYS}d hold from entry_date')

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 3 â€” Overlap management + dynamic position sizing
# MAX_SLOTS = 70th percentile of daily open-position count (unconstrained)
# â†’ 70% of trading days fall below the line, 30% above
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n'+'='*65)
print('STEP 3 â€” Overlap management + dynamic position sizing')
print('='*65)

sim_dates = pd.date_range(START_DATE, END_DATE, freq='B')

events['_absz'] = events['eps_surprise_zscore'].abs()
events = events.sort_values(['entry_date','_absz'], ascending=[True,False]).reset_index(drop=True)
events.drop(columns='_absz', inplace=True)

# â”€â”€ Pass 1: generate all valid trades with no capacity constraint
all_trade_list = []
for _, row in events.iterrows():
    exit_date = tday_offset(row['entry_date'], LONG_HOLD_DAYS)
    if exit_date is None: continue
    all_trade_list.append(row.to_dict() | {'exit_date': exit_date})

all_trades_df = pd.DataFrame(all_trade_list)

# Compute daily concurrent positions (unconstrained)
open_count_raw = pd.Series(0, index=sim_dates)
for _, tr in all_trades_df.iterrows():
    mask = (sim_dates >= tr['entry_date']) & (sim_dates <= tr['exit_date'])
    open_count_raw[mask] += 1

peak_overlap = int(open_count_raw.max())
pct_70       = float(np.percentile(open_count_raw.values, 70))
MAX_SLOTS    = max(5, int(np.ceil(pct_70)))
POS_SIZE     = round(1.0 / MAX_SLOTS, 4)

days_below   = (open_count_raw <= MAX_SLOTS).mean() * 100
print(f'  Unconstrained peak overlap        : {peak_overlap} positions')
print(f'  70th-pct of daily count           : {pct_70:.2f}  â†’  MAX_SLOTS={MAX_SLOTS}')
print(f'  Days at/below MAX_SLOTS           : {days_below:.1f}%  (target â‰¥70%)')
print(f'  POS_SIZE  (1/MAX_SLOTS)           : {POS_SIZE:.4f}  ({POS_SIZE*100:.2f}% per trade)')

# â”€â”€ Pass 2: capacity filter using derived MAX_SLOTS
accepted, rejected_c, open_slots = [], 0, []
for _, row in events.iterrows():
    entry      = row['entry_date']
    open_slots = [ed for ed in open_slots if ed > entry]
    if len(open_slots) >= MAX_SLOTS:
        rejected_c += 1; continue
    exit_date = tday_offset(entry, LONG_HOLD_DAYS)
    if exit_date is None: continue
    open_slots.append(exit_date)
    accepted.append(row.to_dict() | {'exit_date': exit_date})

trades_df = pd.DataFrame(accepted)
print(f'  Signals fired     : {len(events)}')
print(f'  Trades taken      : {len(trades_df)}')
print(f'  Capacity rejected : {rejected_c}')

open_count = pd.Series(0, index=sim_dates)
for _, tr in trades_df.iterrows():
    mask = (sim_dates >= tr['entry_date']) & (sim_dates <= tr['exit_date'])
    open_count[mask] += 1
print(f'  Max simultaneous (accepted) : {open_count.max()}')

fig, ax = plt.subplots(figsize=(12,3))
ax.fill_between(open_count.index, open_count.values, alpha=0.5, color='steelblue')
ax.axhline(MAX_SLOTS, color='red', linestyle='--', lw=1,
           label=f'MAX_SLOTS={MAX_SLOTS} (70th pct) | {POS_SIZE*100:.2f}%/trade | {days_below:.0f}% days below')
ax.set_title(f'Simultaneous Open Positions â€” PEIA Filter | {len(TICKERS)}-stock S&P 500 (70th-pct capacity line)', fontweight='bold')
ax.set_ylabel('Open positions'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_position_overlap.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_position_overlap.png')

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 4 â€” Portfolio simulation
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n'+'='*65)
print('STEP 4 â€” Portfolio simulation (Long-Only)')
print('='*65)
print(f'Long hold: {LONG_HOLD_DAYS}d from entry_date (day+{CONFIRM_DAYS+1}) | MAX_SLOTS: {MAX_SLOTS} | POS_SIZE: {POS_SIZE:.4f} ({POS_SIZE*100:.2f}%)')

TC = TC_BPS / 10_000

px = close_px.reindex(sim_dates, method='ffill')
op = open_px.reindex(sim_dates, method='ffill')

def get_px(frame, date, ticker):
    try:
        v = frame.loc[date, ticker]
        return scalar(v)
    except (KeyError, IndexError):
        return np.nan

port_val      = pd.Series(np.nan, index=sim_dates)
cash          = INIT_CAP
active        = []
trade_results = []

for i, date in enumerate(sim_dates):
    new_today = trades_df[trades_df['entry_date'] == date]
    for _, tr in new_today.iterrows():
        ep = get_px(op, date, tr['ticker'])
        if np.isnan(ep) or ep <= 0: continue
        total_now = cash + sum(p['pos_cash'] for p in active)
        pos_cash  = total_now * POS_SIZE
        tc_entry  = pos_cash * TC
        if pos_cash + tc_entry > cash:
            pos_cash = max(cash - tc_entry, 0)
            if pos_cash <= 0: continue
        cash -= pos_cash + tc_entry
        active.append(dict(
            ticker      = tr['ticker'],
            signal      = tr['signal'],
            exit_date   = tr['exit_date'],
            entry_price = ep,
            pos_cash    = pos_cash,
            z_score     = tr['eps_surprise_zscore'],
            entry_date  = date,
        ))

    still_open = []
    for pos in active:
        if date >= pos['exit_date']:
            cp = get_px(px, date, pos['ticker'])
            if np.isnan(cp) or cp <= 0: cp = pos['entry_price']
            tc_exit = pos['pos_cash'] * TC
            pnl = (cp - pos['entry_price']) / pos['entry_price'] * pos['pos_cash'] - tc_exit
            cash += pos['pos_cash'] + pnl
            trade_results.append(dict(
                ticker      = pos['ticker'],
                entry_date  = pos['entry_date'],
                exit_date   = date,
                entry_price = pos['entry_price'],
                exit_price  = cp,
                pnl         = pnl,
                ret_pct     = pnl / pos['pos_cash'] * 100,
                z_score     = pos['z_score'],
            ))
        else:
            still_open.append(pos)
    active = still_open

    mtm = 0.0
    for pos in active:
        cp = get_px(px, date, pos['ticker'])
        if not (np.isnan(cp) or cp <= 0):
            mtm += (cp - pos['entry_price']) / pos['entry_price'] * pos['pos_cash']

    committed      = sum(p['pos_cash'] for p in active)
    port_val.iloc[i] = cash + committed + mtm

port_val = port_val.ffill()
trade_results_df = pd.DataFrame(trade_results)

print(f'  Trades executed        : {len(trade_results_df)}')
print(f'  Final portfolio value  : ${port_val.iloc[-1]:,.2f}')
print(f'  Total PnL              : ${port_val.iloc[-1] - INIT_CAP:,.2f}')

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 5 â€” Performance metrics
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n'+'='*65)
print('STEP 5 â€” Performance metrics')
print('='*65)

daily_ret = port_val.pct_change().dropna()
n_years   = (port_val.index[-1] - port_val.index[0]).days / 365.25

total_ret = (port_val.iloc[-1] / port_val.iloc[0] - 1) * 100
cagr      = ((port_val.iloc[-1] / port_val.iloc[0]) ** (1 / n_years) - 1) * 100
ann_vol   = daily_ret.std() * np.sqrt(252) * 100
rf_daily  = RF_ANNUAL / 252
sharpe    = (daily_ret.mean() * 252 - RF_ANNUAL) / (daily_ret.std() * np.sqrt(252)) if daily_ret.std() > 0 else 0.0

roll_max = port_val.cummax()
drawdown = (port_val - roll_max) / roll_max * 100
max_dd   = float(drawdown.min())
dd_end   = drawdown.idxmin()
dd_start = port_val[:dd_end].idxmax()
dd_dur   = (dd_end - dd_start).days
calmar   = float(cagr) / abs(max_dd) if max_dd != 0 else np.nan

down_ret = daily_ret[daily_ret < 0]
sortino  = (daily_ret.mean() * 252 - RF_ANNUAL) / (down_ret.std() * np.sqrt(252)) if len(down_ret) > 0 else 0.0

spy_raw = yf.download('SPY', start=START_DATE, end='2026-06-21',
                       auto_adjust=True, progress=False)['Close'].squeeze()
spy_raw.index = pd.to_datetime(spy_raw.index).normalize()
spy = spy_raw.reindex(sim_dates).ffill().bfill()
spy_first = float(spy.dropna().iloc[0])
spy_last  = float(spy.dropna().iloc[-1])
spy_total = (spy_last / spy_first - 1) * 100
spy_cagr  = ((spy_last / spy_first) ** (1 / n_years) - 1) * 100
spy_norm  = spy / spy_first * INIT_CAP
alpha_ann = float(cagr) - spy_cagr

tr       = trade_results_df
wins     = tr[tr['ret_pct'] > 0]
losses   = tr[tr['ret_pct'] <= 0]
win_rate = float(len(wins) / len(tr) * 100) if len(tr) > 0 else 0.0
avg_ret  = float(tr['ret_pct'].mean())
avg_win  = float(wins['ret_pct'].mean())   if len(wins)   > 0 else 0.0
avg_loss = float(losses['ret_pct'].mean()) if len(losses) > 0 else 0.0
gross_p  = float(wins['pnl'].sum())
gross_l  = float(abs(losses['pnl'].sum()))
pfactor  = gross_p / gross_l if gross_l > 0 else np.nan

port_monthly = port_val.resample('ME').last().pct_change().dropna()
spy_monthly  = spy.resample('ME').last().pct_change().dropna()
common_idx   = port_monthly.index.intersection(spy_monthly.index)
pm     = port_monthly.loc[common_idx].values
sm_v   = spy_monthly.loc[common_idx].values
X_ols  = sm.add_constant(sm_v)
ols    = sm.OLS(pm, X_ols).fit()
beta_v = float(ols.params[1])
alpha_monthly = float(ols.params[0])
alpha_ols_ann = float(((1 + alpha_monthly) ** 12 - 1) * 100)
r2_v   = float(ols.rsquared)

print(f"""
  RETURN METRICS
    Total return          : {float(total_ret):>8.2f}%
    CAGR                  : {float(cagr):>8.2f}%
    SPY total return      : {spy_total:>8.2f}%
    SPY CAGR              : {spy_cagr:>8.2f}%
    Alpha vs SPY (ann.)   : {alpha_ann:>8.2f}%

  RISK METRICS
    Annualized volatility : {float(ann_vol):>8.2f}%
    Sharpe ratio          : {float(sharpe):>8.3f}
    Sortino ratio         : {float(sortino):>8.3f}
    Calmar ratio          : {float(calmar):>8.3f}
    Max drawdown          : {max_dd:>8.2f}%
    Drawdown duration     : {dd_dur:>8d} days

  TRADE METRICS
    Total trades          : {len(tr):>8d}
    Win rate              : {win_rate:>8.1f}%
    Avg return / trade    : {avg_ret:>8.2f}%
    Avg winning trade     : {avg_win:>8.2f}%
    Avg losing trade      : {avg_loss:>8.2f}%
    Profit factor         : {pfactor:>8.3f}

  FACTOR EXPOSURE (monthly OLS vs SPY)
    Beta                  : {beta_v:>8.3f}
    R-squared             : {r2_v:>8.3f}
    Alpha (annualized)    : {alpha_ols_ann:>8.2f}%

  Universe: {len(TICKERS)} S&P 500 stocks | Long-Only | PEIA (gap hold + volâ‰¥{VOL_RATIO_MIN}Ã—)
  Entry: day+{CONFIRM_DAYS+1} | Hold: {LONG_HOLD_DAYS}d | MAX_SLOTS={MAX_SLOTS} | POS_SIZE={POS_SIZE:.2%}
  Prior best (499-stock, revision filter): 181 trades, Sharpe=0.322, CAGR=8.37%, MaxDD=-28.06%
  Prior best (100-stock, z>0 + revision): 105 trades, Sharpe=0.585, CAGR=12.73%, MaxDD=-23.11%
""")

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# STEP 6 â€” Visualizations
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('='*65)
print('STEP 6 â€” Visualizations')
print('='*65)

# 6a â€” Equity curve
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(port_val.index, port_val.values, color='steelblue', lw=1.8, label='PEAD Long-Only')
ax.plot(spy_norm.index, spy_norm.values, color='gray',      lw=1.4, ls='--', label='SPY B&H')
ax.axvspan(dd_start, dd_end, alpha=0.12, color='red', label=f'Max DD ({max_dd:.1f}%)')
ax.set_title(
    f'PEAD Long-Only â€” Equity Curve vs SPY  '
    f'(z>{PRIMARY_Z} + PEIA | {LONG_HOLD_DAYS}d hold | {len(TICKERS)}-stock S&P 500 | volâ‰¥{VOL_RATIO_MIN}Ã— | entry day+{CONFIRM_DAYS+1})',
    fontsize=12, fontweight='bold'
)
ax.set_ylabel('Portfolio value ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_equity_curve.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_equity_curve.png')

# 6b â€” Drawdown
fig, ax = plt.subplots(figsize=(13,3.5))
ax.fill_between(drawdown.index, drawdown.values, 0,
                where=drawdown.values < 0, color='crimson', alpha=0.6)
ax.plot(drawdown.index, drawdown.values, color='crimson', lw=0.8)
ax.set_title('Rolling Drawdown from Peak', fontsize=12, fontweight='bold')
ax.set_ylabel('Drawdown (%)'); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_drawdown.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_drawdown.png')

# 6c â€” Trade return distribution
fig, ax = plt.subplots(figsize=(10,4.5))
l_ret = tr['ret_pct'].dropna()
rng   = max(abs(l_ret).max(), 5)
bins  = np.linspace(-rng, rng, 40)
ax.hist(l_ret, bins=bins, color='steelblue', alpha=0.8,
        label=f'Long {LONG_HOLD_DAYS}d ({len(l_ret)} trades)')
ax.axvline(float(l_ret.mean()), color='steelblue', lw=2, ls='--', label=f'Mean {l_ret.mean():.2f}%')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Trade return (%)'); ax.set_ylabel('Count')
ax.set_title(f'Trade Return Distribution â€” Long-Only {LONG_HOLD_DAYS}d hold (z>{PRIMARY_Z} + gap>0)',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_trade_distribution.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_trade_distribution.png')

# 6d â€” Rolling 252-day Sharpe
roll_sh = (daily_ret.rolling(252).mean() - rf_daily) / daily_ret.rolling(252).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(13,4))
ax.plot(roll_sh.index, roll_sh.values, color='darkorange', lw=1.5)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axhline(1, color='green', lw=0.8, ls=':', label='Sharpe = 1')
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values > 0, alpha=0.25, color='green')
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values < 0, alpha=0.25, color='red')
ax.set_title('Rolling 252-Day Sharpe Ratio', fontsize=12, fontweight='bold')
ax.set_ylabel('Sharpe ratio'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_rolling_sharpe.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_rolling_sharpe.png')

# 6e â€” Threshold sensitivity
print('\n  Running threshold sensitivity...')
thresh_rows = []
for thr in THRESHOLDS:
    ev_t = classify_signals(events_raw, thr)
    ev_t = ev_t[ev_t['signal'] != 'NONE'].copy().reset_index(drop=True)
    ev_t['_absz'] = ev_t['eps_surprise_zscore'].abs()
    ev_t = ev_t.sort_values(['entry_date','_absz'], ascending=[True,False]).reset_index(drop=True)
    ev_t.drop(columns='_absz', inplace=True)

    acc_t, rej_t, slots_t = [], 0, []
    for _, row in ev_t.iterrows():
        entry = row['entry_date']
        slots_t = [ed for ed in slots_t if ed > entry]
        if len(slots_t) >= MAX_SLOTS: rej_t += 1; continue
        ex = tday_offset(entry, LONG_HOLD_DAYS)
        if ex is None: continue
        slots_t.append(ex); acc_t.append(row.to_dict() | {'exit_date': ex})

    if not acc_t:
        thresh_rows.append(dict(threshold=thr, trades=0, cagr=0.0, sharpe=0.0, win_rate=0.0, max_dd=0.0))
        continue

    tr_t  = pd.DataFrame(acc_t)
    cash_t = INIT_CAP; act_t = []; pv_t = pd.Series(np.nan, index=sim_dates)
    pv_t.iloc[0] = INIT_CAP; tr_res_t = []

    for i, date in enumerate(sim_dates):
        nw = tr_t[tr_t['entry_date'] == date]
        for _, row in nw.iterrows():
            ep = get_px(op, date, row['ticker'])
            if np.isnan(ep) or ep <= 0: continue
            total_now = cash_t + sum(p['pos_cash'] for p in act_t)
            pc = total_now * POS_SIZE
            tc_e = pc * TC
            if pc + tc_e > cash_t: pc = max(cash_t - tc_e, 0)
            if pc <= 0: continue
            cash_t -= pc + tc_e
            act_t.append(dict(ticker=row['ticker'], signal=row['signal'],
                               exit_date=row['exit_date'], entry_price=ep, pos_cash=pc))
        still_t = []
        for pos in act_t:
            cp = get_px(px, date, pos['ticker'])
            if np.isnan(cp) or cp <= 0: cp = pos['entry_price']
            if date >= pos['exit_date']:
                tc_x = pos['pos_cash'] * TC
                pnl_t = (cp - pos['entry_price']) / pos['entry_price'] * pos['pos_cash'] - tc_x
                cash_t += pos['pos_cash'] + pnl_t
                tr_res_t.append(pnl_t / pos['pos_cash'] * 100)
            else:
                still_t.append(pos)
        act_t = still_t
        mtm_t = 0.0
        for pos in act_t:
            cp = get_px(px, date, pos['ticker'])
            if not (np.isnan(cp) or cp <= 0):
                mtm_t += (cp - pos['entry_price']) / pos['entry_price'] * pos['pos_cash']
        pv_t.iloc[i] = cash_t + sum(p['pos_cash'] for p in act_t) + mtm_t

    pv_t = pv_t.ffill()
    dr_t = pv_t.pct_change().dropna()
    ny_t = (pv_t.index[-1] - pv_t.index[0]).days / 365.25
    cagr_t   = float(((pv_t.iloc[-1]/pv_t.iloc[0])**(1/ny_t)-1)*100)
    sharpe_t = float((dr_t.mean()*252 - RF_ANNUAL)/(dr_t.std()*np.sqrt(252))) if dr_t.std()>0 else 0.0
    mdd_t    = float(((pv_t - pv_t.cummax())/pv_t.cummax()*100).min())
    wr_t     = float((np.array(tr_res_t) > 0).mean()*100) if tr_res_t else 0.0
    thresh_rows.append(dict(threshold=thr, trades=len(tr_res_t),
                            cagr=round(cagr_t,2), sharpe=round(sharpe_t,3),
                            win_rate=round(wr_t,1), max_dd=round(mdd_t,2)))

thresh_df = pd.DataFrame(thresh_rows)
print('\n  Threshold Sensitivity:')
print(thresh_df.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(11,7))
axes = axes.flatten()
for ax, (col, label, color) in zip(axes, [
        ('cagr','CAGR (%)','steelblue'), ('sharpe','Sharpe Ratio','darkorange'),
        ('win_rate','Win Rate (%)','seagreen'), ('max_dd','Max Drawdown (%)','crimson')]):
    ax.bar([str(t) for t in thresh_df['threshold']], thresh_df[col], color=color, alpha=0.8)
    ax.set_title(label, fontweight='bold'); ax.set_xlabel('Z-score threshold')
    ax.grid(True, axis='y', alpha=0.3)
fig.suptitle(f'Threshold Sensitivity â€” PEIA Filter | {len(TICKERS)}-Stock S&P 500', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_threshold_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_threshold_sensitivity.png')

# 6f â€” Per-ticker performance (100 tickers: wider chart, rotated labels)
ticker_perf = (trade_results_df.groupby('ticker')['ret_pct']
               .agg(['mean','count'])
               .rename(columns={'mean':'avg_ret','count':'n_trades'})
               .sort_values('avg_ret'))
fig, ax = plt.subplots(figsize=(24, 6))
colors_bar = ['crimson' if v < 0 else 'steelblue' for v in ticker_perf['avg_ret']]
bars = ax.bar(ticker_perf.index, ticker_perf['avg_ret'], color=colors_bar, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
for bar, (_, row) in zip(bars, ticker_perf.iterrows()):
    h = float(bar.get_height())
    ax.text(bar.get_x()+bar.get_width()/2, h+(0.05 if h>=0 else -0.15),
            f'n={int(row["n_trades"])}', ha='center', va='bottom' if h>=0 else 'top', fontsize=5.5)
ax.set_xlabel('Ticker'); ax.set_ylabel('Avg return per trade (%)')
ax.set_title(
    f'Per-Ticker Average Trade Return  (Long-Only {LONG_HOLD_DAYS}d | z>{PRIMARY_Z} + gap>0 | '
    f'{len(TICKERS)}-stock S&P 500 | revisionâ‰¥1 | entry day+10)',
    fontsize=12, fontweight='bold'
)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR,'pead_per_ticker.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('  Saved: pead_per_ticker.png')

print(f'\nAll outputs saved to: {OUTPUT_DIR}')
print('Done.')


STEP 1 â€” Building earnings event table


  Price data shape: (2147, 499)
Fetching earnings for 499 tickers (3 parallel workers, retry on error)...


  50/499 tickers done â€” 1667 events so far


  100/499 tickers done â€” 3338 events so far


  150/499 tickers done â€” 4957 events so far


...

  z>0.75+gap>0 primary signals   : 1368
    Price+volume confirmed         : 514
    Failed â€” gap faded             : 627
    Failed â€” volume too low        : 227
    No price/vol data              : 0


  z>1.00+gap>0 primary signals   : 1041
    Price+volume confirmed         : 388
    Failed â€” gap faded             : 471
    Failed â€” volume too low        : 182
    No price/vol data              : 0



  Threshold Sensitivity:
 threshold  trades  cagr  sharpe  win_rate  max_dd
      0.00     586  8.88   0.339      59.2  -33.05
      0.25     556 10.96   0.511      61.9  -19.97
      0.50     491 13.14   0.706      64.2  -16.71
      0.75     391 10.78   0.616      64.5  -11.82
      1.00     320  8.35   0.458      65.0  -11.59


  Saved: pead_threshold_sensitivity.png


  Saved: pead_per_ticker.png

All outputs saved to: C:\Trading Algo\stocks_sentiment_PEAD\outputs
Done.


In [ ]:
# H1 DIAGNOSTIC -- z-ranked capacity queue vs FIFO ordering
# The main backtest already sorts by z-score descending within each entry_date
# before the capacity allocation pass. This cell quantifies exactly how much
# that sort contributes by running a FIFO counterfactual (ticker-alphabetical
# within the same date, neutral ordering) and comparing key metrics side by side.
print('='*65)
print('H1 DIAGNOSTIC -- z-ranked queue vs FIFO')
print('='*65)

zcol = 'eps_surprise_zscore'

# z-score split: which signals were accepted under the existing H1 ordering?
acc_set = set(zip(trades_df['ticker'], trades_df['entry_date'].astype(str)))
ev_cpy  = events.copy()
in_acc  = ev_cpy.apply(
    lambda r: (r['ticker'], str(r['entry_date'])) in acc_set, axis=1)
acc_ev  = ev_cpy[in_acc]
rej_ev  = ev_cpy[~in_acc]

az_acc = acc_ev[zcol]
az_rej = rej_ev[zcol]
print(f'
  z-score split (H1 z-ranked run):')
print(f'    Accepted ({len(acc_ev):3d})  avg={az_acc.mean():.4f}  '
      f'p25/p50/p75 = {az_acc.quantile(.25):.3f} / {az_acc.median():.3f} / {az_acc.quantile(.75):.3f}')
print(f'    Rejected ({len(rej_ev):3d})  avg={az_rej.mean():.4f}  '
      f'p25/p50/p75 = {az_rej.quantile(.25):.3f} / {az_rej.median():.3f} / {az_rej.quantile(.75):.3f}')
print(f'    z-lift from ranking  : +{az_acc.mean() - az_rej.mean():.4f}')

# FIFO counterfactual: entry_date ASC, ticker ASC (alphabetical = neutral within same date)
ev_fifo = events.sort_values(
    ['entry_date', 'ticker'], ascending=[True, True]).reset_index(drop=True)

acc_f, slots_f = [], []
for _, row in ev_fifo.iterrows():
    entry   = row['entry_date']
    slots_f = [ed for ed in slots_f if ed > entry]
    if len(slots_f) >= MAX_SLOTS:
        continue
    ex = tday_offset(entry, LONG_HOLD_DAYS)
    if ex is None:
        continue
    slots_f.append(ex)
    acc_f.append(row.to_dict() | {'exit_date': ex})

trades_fifo = pd.DataFrame(acc_f)
TR_r = TC_BPS / 10_000

# Full portfolio simulation for FIFO ordering
cash_f = INIT_CAP; active_f = []; tr_f = []
pv_f   = pd.Series(np.nan, index=sim_dates)

for i_f, date in enumerate(sim_dates):
    for _, tr in trades_fifo[trades_fifo['entry_date'] == date].iterrows():
        ep = get_px(op, date, tr['ticker'])
        if np.isnan(ep) or ep <= 0:
            continue
        tot = cash_f + sum(p['pos_cash'] for p in active_f)
        pc  = tot * POS_SIZE
        tc  = pc * TR_r
        if pc + tc > cash_f:
            pc = max(cash_f - tc, 0)
        if pc <= 0:
            continue
        cash_f -= pc + tc
        active_f.append({'ticker': tr['ticker'], 'exit_date': tr['exit_date'],
                          'entry_price': ep, 'pos_cash': pc})
    still_f = []
    for pos in active_f:
        cp  = get_px(px, date, pos['ticker'])
        if np.isnan(cp) or cp <= 0:
            cp = pos['entry_price']
        ret = (cp - pos['entry_price']) / pos['entry_price']
        if date >= pos['exit_date']:
            cash_f += pos['pos_cash'] * (1 + ret) - pos['pos_cash'] * TR_r
            tr_f.append(ret * 100)
        else:
            still_f.append(pos)
    active_f = still_f
    mtm_f = 0.0
    for pos in active_f:
        cp_m = get_px(px, date, pos['ticker'])
        if not (np.isnan(cp_m) or cp_m <= 0):
            mtm_f += (cp_m - pos['entry_price']) / pos['entry_price'] * pos['pos_cash']
    pv_f.iloc[i_f] = cash_f + sum(p['pos_cash'] for p in active_f) + mtm_f

pv_f   = pv_f.ffill()
dr_f   = pv_f.pct_change().dropna()
ny_f   = (pv_f.index[-1] - pv_f.index[0]).days / 365.25
cagr_f = ((pv_f.iloc[-1] / pv_f.iloc[0]) ** (1 / ny_f) - 1) * 100
dn_f   = dr_f[dr_f < 0]
sort_f = (dr_f.mean()*252 - RF_ANNUAL) / (dn_f.std()*np.sqrt(252)) if len(dn_f) > 0 else 0.0
shp_f  = (dr_f.mean()*252 - RF_ANNUAL) / (dr_f.std()*np.sqrt(252)) if dr_f.std() > 0 else 0.0
mdd_f  = float(((pv_f - pv_f.cummax()) / pv_f.cummax() * 100).min())
tr_arr = np.array(tr_f)
wr_f   = float((tr_arr > 0).mean() * 100) if len(tr_arr) > 0 else 0.0
zf_avg = trades_fifo[zcol].mean()

print(f'
  FIFO run: Accepted ({len(trades_fifo):3d}) avg_z={zf_avg:.4f}')

n_sw = len(
    set(zip(trades_df['ticker'],
            trades_df['entry_date'].astype(str))).symmetric_difference(
    set(zip(trades_fifo['ticker'],
            trades_fifo['entry_date'].astype(str)))))

# Comparison table
h1z  = az_acc.mean()
sep  = '='*65
sepm = '-'*65
print(f'
{sep}')
print('  {:<22}  {:>12}  {:>10}  {:>9}'.format('Metric', 'z-Ranked', 'FIFO', 'Delta'))
print(sepm)
for lbl, h1v, fiv in [
    ('CAGR (%)',         float(cagr),     cagr_f),
    ('Sortino',          float(sortino),  sort_f),
    ('Sharpe',           float(sharpe),   shp_f),
    ('Max DD (%)',       float(max_dd),   mdd_f),
    ('Win Rate (%)',     float(win_rate), wr_f),
    ('Avg z (accepted)', h1z,            zf_avg),
]:
    d = h1v - fiv
    print(f'  {lbl:<22}  {h1v:>12.3f}  {fiv:>10.3f}  {d:>+9.3f}')
print(sepm)
print(f'  Trade-set swaps (H1 vs FIFO) : {n_sw}')
print(sep)


In [3]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# PEAD PARAMETER SWEEP â€” 72 Combinations via VectorBT
# Dimensions: Z-threshold (3) Ã— Gap filter (2) Ã— Long hold (4) Ã— Short hold (3)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
import vectorbt as vbt
import itertools

TC = TC_BPS / 10_000   # one-way TC fraction (5 bps)

# â”€â”€ Parameter grid â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
Z_THRESHOLDS = [0.0, 0.5, 1.0]
GAP_FILTERS  = [True, False]          # True = require gap in signal direction
LONG_HOLDS   = [7, 21, 42, 63]       # calendar days
SHORT_HOLDS  = [7, 14, 21]           # calendar days

combos = list(itertools.product(Z_THRESHOLDS, GAP_FILTERS, LONG_HOLDS, SHORT_HOLDS))
combo_labels = [f"z{z:.1f}_g{int(g)}_l{l:02d}_s{s:02d}" for z, g, l, s in combos]
n_combos = len(combos)

print('='*65)
print(f"PARAMETER SWEEP â€” {n_combos} combinations")
print('='*65)
print(f"  Z thresholds : {Z_THRESHOLDS}")
print(f"  Gap filter   : {GAP_FILTERS}  (True = require gap in signal direction)")
print(f"  Long holds   : {LONG_HOLDS} calendar days")
print(f"  Short holds  : {SHORT_HOLDS} calendar days")

# â”€â”€ Build signal arrays (O(1) date lookup via dict) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
n_dates    = len(sim_dates)
n_tickers  = len(TICKERS)
date_idx   = {d: i for i, d in enumerate(sim_dates)}
ticker_idx = {t: i for i, t in enumerate(TICKERS)}

# shape: (n_combos, n_tickers, n_dates)
long_entry_arr  = np.zeros((n_combos, n_tickers, n_dates), dtype=bool)
long_exit_arr   = np.zeros((n_combos, n_tickers, n_dates), dtype=bool)
short_entry_arr = np.zeros((n_combos, n_tickers, n_dates), dtype=bool)
short_exit_arr  = np.zeros((n_combos, n_tickers, n_dates), dtype=bool)

print("\nBuilding signal matrices...", end='', flush=True)
for ci, (z_thr, use_gap, l_hold, s_hold) in enumerate(combos):
    for ticker in TICKERS:
        ti   = ticker_idx[ticker]
        ev_t = events_raw[events_raw['ticker'] == ticker]

        # â”€â”€ Long: z > z_thr (+ gap > 0 if gap filter on)
        lm = ev_t['eps_surprise_zscore'] > z_thr
        if use_gap:
            lm &= ev_t['gap_pct'] > 0
        for _, row in ev_t[lm].iterrows():
            e = row['trade_start_date']
            x = tday_offset(e, l_hold)
            if e in date_idx:
                long_entry_arr[ci, ti, date_idx[e]] = True
            if x is not None and x in date_idx:
                long_exit_arr[ci, ti, date_idx[x]] = True

        # â”€â”€ Short: z < -z_thr (+ gap < 0 if gap filter on)
        sm = ev_t['eps_surprise_zscore'] < -z_thr
        if use_gap:
            sm &= ev_t['gap_pct'] < 0
        for _, row in ev_t[sm].iterrows():
            e = row['trade_start_date']
            x = tday_offset(e, s_hold)
            if e in date_idx:
                short_entry_arr[ci, ti, date_idx[e]] = True
            if x is not None and x in date_idx:
                short_exit_arr[ci, ti, date_idx[x]] = True

print(" done.")
print(f"  Long entry signals  : {long_entry_arr.sum():,}")
print(f"  Short entry signals : {short_entry_arr.sum():,}")

# â”€â”€ Stack into DataFrames: (n_dates, n_combos Ã— n_tickers) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
mi_cols = pd.MultiIndex.from_product([combo_labels, TICKERS], names=['combo', 'ticker'])

def _to_df(arr, dtype=bool):
    """(n_combos, n_tickers, n_dates) â†’ (n_dates, n_combos Ã— n_tickers) DataFrame."""
    return pd.DataFrame(
        arr.reshape(n_combos * n_tickers, n_dates).T.astype(dtype),
        index=sim_dates, columns=mi_cols
    )

close_sim   = close_px[TICKERS].reindex(sim_dates, method='ffill')
# Tile prices: same 20-ticker matrix repeated for each of 72 combos
price_stack = pd.DataFrame(
    np.tile(close_sim.values, (1, n_combos)),
    index=sim_dates, columns=mi_cols
)

long_entry_df  = _to_df(long_entry_arr)
long_exit_df   = _to_df(long_exit_arr)
short_entry_df = _to_df(short_entry_arr)
short_exit_df  = _to_df(short_exit_arr)

# â”€â”€ VectorBT: two passes (long-only / short-only), SizeType.Value â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# SizeType.Percent is incompatible with combined long+short signals in VBT 1.0.
# Solution: run long and short portfolios separately, combine daily returns.
group_by   = mi_cols.get_level_values('combo')
trade_size = INIT_CAP * POS_SIZE          # fixed $20K per trade (approx 20% of $100K)
half_cash  = INIT_CAP / 2                 # split initial capital equally between L and S

print(f"\nRunning VBT long-side sweep (1440 columns)...", flush=True)
pf_long = vbt.Portfolio.from_signals(
    close     = price_stack,
    entries   = long_entry_df,
    exits     = long_exit_df,
    size      = trade_size,
    size_type = 'Value',
    fees      = TC,
    init_cash = half_cash,
    group_by  = group_by,
    cash_sharing = True,
    freq      = 'D',
)

print(f"Running VBT short-side sweep (1440 columns)...", flush=True)
pf_short = vbt.Portfolio.from_signals(
    close         = price_stack,
    short_entries = short_entry_df,
    short_exits   = short_exit_df,
    size          = trade_size,
    size_type     = 'Value',
    fees          = TC,
    init_cash     = half_cash,
    group_by      = group_by,
    cash_sharing  = True,
    freq          = 'D',
)
print("VBT sweep complete.")

# Combine: weight-average returns (each half contributes equally to portfolio)
# combined_ret[t] = 0.5 * ret_long[t] + 0.5 * ret_short[t]
rets_long  = pf_long.returns()    # (n_dates, n_combos)
rets_short = pf_short.returns()   # (n_dates, n_combos)

# â”€â”€ Extract per-combo performance metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Extracting per-combo stats...", flush=True)
# Combined daily return series for each combo
rets_df = 0.5 * rets_long + 0.5 * rets_short

results_rows = []
for cid, (z_thr, use_gap, l_hold, s_hold) in zip(combo_labels, combos):
    r = rets_df[cid].dropna()
    if len(r) < 50:
        results_rows.append(dict(
            z_thr=z_thr, use_gap=use_gap, l_hold=l_hold, s_hold=s_hold, combo=cid,
            n_long=0, n_short=0, total_trades=0,
            total_return=np.nan, cagr=np.nan, sharpe=np.nan,
            max_dd=np.nan, ann_vol=np.nan, win_rate=np.nan
        ))
        continue

    n_yr      = (r.index[-1] - r.index[0]).days / 365.25
    tot_ret   = float((1 + r).prod() - 1)
    cagr_v    = float((1 + tot_ret) ** (1 / n_yr) - 1) * 100 if (n_yr > 0 and tot_ret > -1) else -100.0
    ann_vol_v = float(r.std() * np.sqrt(252)) * 100
    sharpe_v  = (
        float((r.mean() * 252 - RF_ANNUAL) / (r.std() * np.sqrt(252)))
        if r.std() > 0 else 0.0
    )
    cum      = (1 + r).cumprod()
    max_dd_v = float(((cum - cum.cummax()) / cum.cummax()).min()) * 100

    # per-combo trade stats from VBT (long + short combined)
    try:
        sl = pf_long[cid].stats()
        ss = pf_short[cid].stats()
        nt_l     = int(sl.get('Total Closed Trades', 0))
        nt_s     = int(ss.get('Total Closed Trades', 0))
        wr_l     = float(sl.get('Win Rate [%]', np.nan))
        wr_s     = float(ss.get('Win Rate [%]', np.nan))
        n_trades = nt_l + nt_s
        # weighted win rate (by trade count)
        if (nt_l + nt_s) > 0:
            win_rate = (wr_l * nt_l + wr_s * nt_s) / (nt_l + nt_s)
        else:
            win_rate = np.nan
    except Exception:
        win_rate = np.nan
        n_trades = int(long_entry_df[cid].values.sum() + short_entry_df[cid].values.sum())

    n_l = int(long_entry_df[cid].values.sum())
    n_s = int(short_entry_df[cid].values.sum())

    results_rows.append(dict(
        z_thr=z_thr, use_gap=use_gap, l_hold=l_hold, s_hold=s_hold, combo=cid,
        n_long=n_l, n_short=n_s, total_trades=n_trades,
        total_return = round(tot_ret * 100, 2),
        cagr         = round(cagr_v,    2),
        sharpe       = round(sharpe_v,  3),
        max_dd       = round(max_dd_v,  2),
        ann_vol      = round(ann_vol_v, 2),
        win_rate     = round(win_rate,  1),
    ))

results_df       = pd.DataFrame(results_rows)
results_df_sorted = results_df.sort_values('sharpe', ascending=False).reset_index(drop=True)

print(f"\n{'='*72}")
print(f"TOP 20 COMBINATIONS BY SHARPE RATIO")
print(f"{'='*72}")
disp = results_df_sorted[
    ['z_thr','use_gap','l_hold','s_hold','total_trades','cagr','sharpe','max_dd','win_rate']
].head(20).copy()
disp.columns = ['Z-thr','Gap','L-hold','S-hold','Trades','CAGR%','Sharpe','MaxDD%','WinRate%']
print(disp.to_string(index=False))

# â”€â”€ Visualisations â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('PEAD Parameter Sweep â€” 72 Combinations', fontsize=14, fontweight='bold', y=1.01)

# â”€â”€ Plot 1: Sharpe heatmap â€” Z-threshold Ã— Long-hold â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax = axes[0, 0]
pivot = results_df.groupby(['z_thr', 'l_hold'])['sharpe'].mean().unstack('l_hold')
sns.heatmap(
    pivot, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
    linewidths=0.5, cbar_kws={'label': 'Mean Sharpe'}
)
ax.set_title('Mean Sharpe: Z-threshold Ã— Long Hold\n(avg over gap filter & short hold)', fontweight='bold')
ax.set_xlabel('Long Hold (days)'); ax.set_ylabel('Z-score Threshold')

# â”€â”€ Plot 2: Sharpe heatmap â€” Short-hold Ã— Gap-filter â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax = axes[0, 1]
pivot2 = results_df.groupby(['s_hold', 'use_gap'])['sharpe'].mean().unstack('use_gap')
pivot2.columns = ['Gap OFF', 'Gap ON']
sns.heatmap(
    pivot2, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
    linewidths=0.5, cbar_kws={'label': 'Mean Sharpe'}
)
ax.set_title('Mean Sharpe: Short Hold Ã— Gap Filter\n(avg over z-threshold & long hold)', fontweight='bold')
ax.set_xlabel('Gap Filter'); ax.set_ylabel('Short Hold (days)')

# â”€â”€ Plot 3: Top-15 combos horizontal bar â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax = axes[1, 0]
top15 = results_df_sorted.head(15).copy()
top15['label'] = [
    f"z{r.z_thr:.1f} gap={'Y' if r.use_gap else 'N'} L{r.l_hold:02d}d S{r.s_hold:02d}d"
    for _, r in top15.iterrows()
]
# reverse so best is at top
labels_rev  = top15['label'].values[::-1]
sharpe_rev  = top15['sharpe'].values[::-1]
colors      = ['steelblue' if s >= 0 else 'crimson' for s in sharpe_rev]
bars        = ax.barh(range(len(top15)), sharpe_rev, color=colors, alpha=0.85)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(labels_rev, fontsize=8.5)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Sharpe Ratio')
ax.set_title('Top 15 Combinations by Sharpe', fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
for bar, idx in zip(bars, top15.index[::-1]):
    r = top15.loc[idx]
    w = bar.get_width()
    ax.text(
        max(w + 0.003, 0.003), bar.get_y() + bar.get_height() / 2,
        f"CAGR={r.cagr:.1f}%  DD={r.max_dd:.1f}%",
        va='center', fontsize=7
    )

# â”€â”€ Plot 4: CAGR vs Max-Drawdown scatter, coloured by Sharpe â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax = axes[1, 1]
valid = results_df.dropna(subset=['cagr', 'max_dd', 'sharpe'])
vlo   = valid['sharpe'].quantile(0.10)
vhi   = valid['sharpe'].quantile(0.90)
sc = ax.scatter(
    valid['max_dd'], valid['cagr'],
    c=valid['sharpe'], cmap='RdYlGn', s=55, alpha=0.85,
    edgecolors='gray', lw=0.3, vmin=vlo, vmax=vhi
)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
for _, r in results_df_sorted.head(5).iterrows():
    ax.annotate(
        f"z{r.z_thr:.1f} L{r.l_hold} S{r.s_hold}",
        xy=(r.max_dd, r.cagr), xytext=(4, 4), textcoords='offset points', fontsize=7.5
    )
ax.set_xlabel('Max Drawdown (%)')
ax.set_ylabel('CAGR (%)')
ax.set_title('CAGR vs Max Drawdown\n(coloured by Sharpe; top-5 labeled)', fontweight='bold')
ax.grid(True, alpha=0.3)

fig.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'pead_sweep_results.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"\nSaved: {out_path}")

# â”€â”€ Print best combination â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best = results_df_sorted.iloc[0]
print(f"\n{'='*60}")
print(f"OPTIMAL COMBINATION  (by Sharpe)")
print(f"{'='*60}")
print(f"  Z-threshold  : {best.z_thr}")
print(f"  Gap filter   : {best.use_gap}")
print(f"  Long hold    : {best.l_hold} calendar days")
print(f"  Short hold   : {best.s_hold} calendar days")
print(f"  CAGR         : {best.cagr:.2f}%")
print(f"  Sharpe       : {best.sharpe:.3f}")
print(f"  Max DD       : {best.max_dd:.2f}%")
print(f"  Win Rate     : {best.win_rate:.1f}%")
print(f"  Trade count  : {best.total_trades}")
print(f"{'='*60}")
print(f"\nAll sweep results: {len(results_df)} rows in results_df")
print(f"Sorted by Sharpe: results_df_sorted")


PARAMETER SWEEP â€” 72 combinations
  Z thresholds : [0.0, 0.5, 1.0]
  Gap filter   : [True, False]  (True = require gap in signal direction)
  Long holds   : [7, 21, 42, 63] calendar days
  Short holds  : [7, 14, 21] calendar days

Building signal matrices...

 done.
  Long entry signals  : 213,324


  Short entry signals : 248,328



Running VBT long-side sweep (1440 columns)...


Running VBT short-side sweep (1440 columns)...


VBT sweep complete.


Extracting per-combo stats...



TOP 20 COMBINATIONS BY SHARPE RATIO
 Z-thr   Gap  L-hold  S-hold  Trades   CAGR%  Sharpe  MaxDD%  WinRate%
   0.0 False      42      14    9286 -100.00   0.272 -913.09      43.9
   0.0 False      21      14    9880 -100.00   0.271 -939.16      44.4
   0.0 False      63      14    8900 -100.00   0.270 -929.14      43.1
   0.0 False       7      14   10823 -100.00   0.269 -906.48      44.3
   0.0 False      42       7    9621  -54.97   0.157 -128.10      45.5
   0.0 False      21       7   10215  -53.03   0.155 -129.00      45.9
   0.0 False      63       7    9235  -54.30   0.152 -128.13      44.8
   0.0 False       7       7   11158  -56.51   0.145 -127.82      45.7
   1.0 False      42       7    1840  -60.05   0.117 -104.04      49.8
   1.0 False      21       7    1951  -60.42   0.110 -103.76      50.0
   1.0 False      63       7    1568  -61.16   0.109 -103.84      46.7
   1.0 False       7       7    2345  -61.26   0.097 -103.20      48.9
   1.0  True      42      14    1233 -10


Saved: C:\Trading Algo\stocks_sentiment_PEAD\outputs\pead_sweep_results.png

OPTIMAL COMBINATION  (by Sharpe)
  Z-threshold  : 0.0
  Gap filter   : False
  Long hold    : 42 calendar days
  Short hold   : 14 calendar days
  CAGR         : -100.00%
  Sharpe       : 0.272
  Max DD       : -913.09%
  Win Rate     : 43.9%
  Trade count  : 9286

All sweep results: 72 rows in results_df
Sorted by Sharpe: results_df_sorted


## Data Layer
### Load HuggingFace and verify coverage

In [4]:
# â”€â”€ S&P 500 Earnings Call Transcripts â€” fast path via HF Datasets Server API â”€â”€
import requests, pandas as pd, io

DATASET = "kurry/sp500_earnings_transcripts"
API     = "https://datasets-server.huggingface.co"
TIMEOUT = 30

def api_get(endpoint, **params):
    r = requests.get(f"{API}/{endpoint}", params={"dataset": DATASET, **params}, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()

# â”€â”€ 1 & 2. Total rows + column names â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# No config param â†’ response is nested under ["dataset_info"]["default"]
info = api_get("info")["dataset_info"]["default"]
total = info["splits"]["train"]["num_examples"]
cols  = list(info["features"].keys())

print(f"Dataset size (total rows): {total:,}")
print(f"\nColumn names: {cols}")

# â”€â”€ 4. Sample 5 rows (pre-computed, returns instantly) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rows = api_get("first-rows", config="default", split="train")["rows"][:5]
sample_df = pd.DataFrame([r["row"] for r in rows])
print("\nSample (5 rows):")
print(sample_df.to_string())

# â”€â”€ Download first parquet shard â€” ticker + date columns only â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
parquet_files = api_get("parquet", config="default")["parquet_files"]
shard_url = parquet_files[0]["url"]
print(f"\nDownloading shard: {shard_url.split('/')[-1]} ...")

r = requests.get(shard_url, timeout=120)
r.raise_for_status()
# Read only lightweight columns (skips heavy 'content' / 'structured_content')
shard = pd.read_parquet(io.BytesIO(r.content), columns=["symbol", "date", "year", "quarter"])

# â”€â”€ 3. Date range â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
shard["date"] = pd.to_datetime(shard["date"], errors="coerce")
print(f"\nDate range: {shard['date'].min().date()} â†’ {shard['date'].max().date()}")

# â”€â”€ 5 & 6. Unique tickers + presence check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
tickers_upper = shard["symbol"].dropna().str.upper()
print(f"\nUnique tickers: {tickers_upper.nunique():,}")

watch_list = ["NVDA", "MSFT", "META", "GOOGL", "AMZN", "JPM", "TSLA"]
present = set(tickers_upper.unique())
print("\nTicker presence check:")
for t in watch_list:
    print(f"  {t}: {'found' if t in present else 'not found'}")


Dataset size (total rows): 33,362

Column names: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id']

Sample (5 rows):
  symbol  quarter  year                 date                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [ ]:
# â”€â”€ Stage 2: Filter & parse earnings transcripts â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import requests, io, json, os, time
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASET   = "kurry/sp500_earnings_transcripts"
API       = "https://datasets-server.huggingface.co"
OUT_DIR   = "data"
OUT_FILE  = os.path.join(OUT_DIR, "transcripts_filtered.parquet")
TIMEOUT   = 300

SYMBOLS = {
    "NVDA","AMD","MSFT","META","GOOGL","AMZN","TSLA",
    "NKE","MCD","BKNG","NFLX","SPOT","DIS",
    "JPM","GS","UNH","LLY","CAT","XOM","CRM"
}
YEAR_MIN, YEAR_MAX = 2018, 2024

os.makedirs(OUT_DIR, exist_ok=True)

# â”€â”€ 1. Fetch shard URLs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
r = requests.get(f"{API}/parquet", params={"dataset": DATASET, "config": "default"}, timeout=30)
r.raise_for_status()
shard_urls = [f["url"] for f in r.json()["parquet_files"]]
print(f"Shards to download: {len(shard_urls)}")

# â”€â”€ 2. Download shards in parallel with progress â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def download_shard(url):
    name = url.split("/")[-1]
    t0 = time.time()
    resp = requests.get(url, timeout=TIMEOUT, stream=True)
    resp.raise_for_status()
    buf = io.BytesIO()
    downloaded = 0
    for chunk in resp.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
        buf.write(chunk)
        downloaded += len(chunk)
    buf.seek(0)
    elapsed = time.time() - t0
    print(f"  {name}: {downloaded/1e6:.1f} MB in {elapsed:.1f}s")
    return buf

print("Downloading shards (parallel)...")
buffers = [None] * len(shard_urls)
with ThreadPoolExecutor(max_workers=len(shard_urls)) as pool:
    futures = {pool.submit(download_shard, url): i for i, url in enumerate(shard_urls)}
    for fut in as_completed(futures):
        idx = futures[fut]
        buffers[idx] = fut.result()

# â”€â”€ 3. Read with pyarrow predicate pushdown (filter before pandas load) â”€â”€â”€â”€â”€â”€â”€
filters = [
    ("symbol", "in", pa.array(list(SYMBOLS))),
    ("year",   ">=", YEAR_MIN),
    ("year",   "<=", YEAR_MAX),
]
print("\nApplying filters...")
frames = []
for buf in buffers:
    table = pq.read_table(buf, filters=filters)
    frames.append(table.to_pandas())

df = pd.concat(frames, ignore_index=True)
print(f"Rows after filter: {len(df):,}")

# â”€â”€ 4. Parse structured_content â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def split_transcript(row):
    sc = row.get("structured_content") or []
    content = row.get("content", "")

    # structured_content may already be a list or a JSON string
    if isinstance(sc, str):
        try:
            sc = json.loads(sc)
        except (json.JSONDecodeError, TypeError):
            sc = []

    if not sc:
        return content, "", len(content.split())

    # Identify first 5 unique speakers (prepared remarks speakers)
    seen, prep_speakers = [], set()
    for turn in sc:
        spk = turn.get("speaker", "")
        if spk and spk not in seen:
            seen.append(spk)
        if len(seen) >= 5:
            break
    prep_speakers = set(seen)

    # Split at first turn whose speaker is outside the prep set
    split_idx = len(sc)
    for i, turn in enumerate(sc):
        if turn.get("speaker", "") not in prep_speakers:
            split_idx = i
            break

    prep_turns = sc[:split_idx]
    qa_turns   = sc[split_idx:]

    prepared_remarks = " ".join(t.get("text", "") for t in prep_turns).strip()
    qa_text          = " ".join(t.get("text", "") for t in qa_turns).strip()
    word_count       = len(content.split())

    return prepared_remarks, qa_text, word_count

print("Parsing structured_content...")
parsed = df.apply(split_transcript, axis=1, result_type="expand")
parsed.columns = ["prepared_remarks", "qa_text", "word_count"]
df = pd.concat([df, parsed], axis=1)

# â”€â”€ 5. Save â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df.to_parquet(OUT_FILE, index=False)
print(f"\nSaved to {OUT_FILE}")

# â”€â”€ 6. Summary stats â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"\nTotal rows saved: {len(df):,}")
print(f"\nRows per ticker:")
print(df.groupby("symbol").size().sort_values(ascending=False).to_string())
print(f"\nRows with non-empty qa_text: {(df['qa_text'].str.len() > 0).sum():,}")
print(f"\nDate range per ticker:")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
date_range = df.groupby("symbol")["date"].agg(["min","max"])
date_range.columns = ["earliest", "latest"]
print(date_range.sort_index().to_string())


### Data Filtration for 20 tickers from 2018-2024

In [ ]:
# â”€â”€ Stage 2a: Download â€” targeted range-request fetch â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Run once. Saves raw filtered rows to data/transcripts_raw.parquet.
# Re-run only if you change SYMBOLS or YEAR range.

import io, requests, json, os, math, time
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASET  = "kurry/sp500_earnings_transcripts"
API      = "https://datasets-server.huggingface.co"
RAW_FILE = "data/transcripts_raw.parquet"
SYMBOLS  = {
    "NVDA","AMD","MSFT","META","GOOGL","AMZN","TSLA",
    "NKE","MCD","BKNG","NFLX","SPOT","DIS",
    "JPM","GS","UNH","LLY","CAT","XOM","CRM",
}
YEAR_MIN, YEAR_MAX  = 2018, 2024
READ_COLS           = ["symbol", "quarter", "year", "date", "structured_content"]
SCOUT_WORKERS       = 10
FETCH_WORKERS       = 8
SMALL_THRESHOLD     = 100 * 1024 * 1024

os.makedirs("data", exist_ok=True)

class RangeFile(io.RawIOBase):
    def __init__(self, cdn_url, session, size):
        super().__init__()
        self._url, self._session, self._size, self._pos = cdn_url, session, size, 0
    def readable(self):  return True
    def seekable(self):  return True
    def writable(self):  return False
    def readinto(self, b):
        data = self._fetch(len(b)); n = len(data); b[:n] = data; return n
    def read(self, n=-1): return self._fetch(n)
    def _fetch(self, n):
        if n == 0: return b""
        end = (self._size - 1) if n < 0 else min(self._pos + n - 1, self._size - 1)
        if self._pos > end: return b""
        data = self._session.get(self._url,
                                  headers={"Range": f"bytes={self._pos}-{end}"},
                                  timeout=120).content
        self._pos = end + 1
        return data
    def seek(self, pos, whence=0):
        if   whence == 0: self._pos = pos
        elif whence == 1: self._pos += pos
        elif whence == 2: self._pos = self._size + pos
        return self._pos
    def tell(self): return self._pos

def resolve_cdn(hf_url):
    s = requests.Session()
    r = s.head(hf_url, allow_redirects=True, timeout=20)
    return r.url, int(r.headers["Content-Length"])

def make_pf(cdn_url, size):
    return pq.ParquetFile(RangeFile(cdn_url, requests.Session(), size))

def scout_chunk(rg_indices, cdn_url, size):
    pf = make_pf(cdn_url, size)
    hits = []
    for i in rg_indices:
        df = pf.read_row_group(i, columns=["symbol", "year"]).to_pandas()
        if (df["symbol"].isin(SYMBOLS) & df["year"].between(YEAR_MIN, YEAR_MAX)).any():
            hits.append(i)
    return hits

def fetch_chunk(rg_indices, cdn_url, size):
    pf = make_pf(cdn_url, size)
    frames = []
    for i in rg_indices:
        df   = pf.read_row_group(i, columns=READ_COLS).to_pandas()
        mask = df["symbol"].isin(SYMBOLS) & df["year"].between(YEAR_MIN, YEAR_MAX)
        if mask.any(): frames.append(df[mask])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=READ_COLS)

def process_large_shard(hf_url, name):
    cdn_url, size = resolve_cdn(hf_url)
    n_rgs = make_pf(cdn_url, size).metadata.num_row_groups
    print(f"  {name}: {size/1e6:.0f} MB, {n_rgs} row groups")
    t0 = time.time()
    batch_size = math.ceil(n_rgs / SCOUT_WORKERS)
    batches = [list(range(i, min(i+batch_size, n_rgs))) for i in range(0, n_rgs, batch_size)]
    target_rgs = []
    with ThreadPoolExecutor(max_workers=SCOUT_WORKERS) as pool:
        for f in as_completed([pool.submit(scout_chunk, b, cdn_url, size) for b in batches]):
            target_rgs.extend(f.result())
    target_rgs.sort()
    print(f"  Scout: {time.time()-t0:.1f}s â€” {len(target_rgs)}/{n_rgs} row groups matched")
    t1 = time.time()
    batch_size = max(1, math.ceil(len(target_rgs) / FETCH_WORKERS))
    batches = [target_rgs[i:i+batch_size] for i in range(0, len(target_rgs), batch_size)]
    frames = []
    with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as pool:
        for f in as_completed([pool.submit(fetch_chunk, b, cdn_url, size) for b in batches]):
            df = f.result()
            if not df.empty: frames.append(df)
    print(f"  Fetch: {time.time()-t1:.1f}s")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=READ_COLS)

def download_small_shard(hf_url, name):
    cdn_url, size = resolve_cdn(hf_url)
    print(f"  {name}: {size/1e6:.0f} MB â€” full download")
    t0 = time.time()
    resp = requests.get(cdn_url, timeout=300); resp.raise_for_status()
    tbl = pq.read_table(io.BytesIO(resp.content), columns=READ_COLS,
                        filters=[("symbol", "in", pa.array(sorted(SYMBOLS))),
                                 ("year", ">=", YEAR_MIN), ("year", "<=", YEAR_MAX)])
    print(f"  Done: {time.time()-t0:.1f}s")
    return tbl.to_pandas()

shards = requests.get(f"{API}/parquet",
                      params={"dataset": DATASET, "config": "default"},
                      timeout=30).json()["parquet_files"]
print(f"Shards: {len(shards)}")
all_frames = []
for shard in shards:
    url, name, size = shard["url"], shard["url"].split("/")[-1], shard["size"]
    print(f"\nProcessing {name} ({size/1e6:.0f} MB)...")
    all_frames.append(process_large_shard(url, name) if size > SMALL_THRESHOLD
                      else download_small_shard(url, name))

df_raw = pd.concat(all_frames, ignore_index=True)
df_raw.to_parquet(RAW_FILE, index=False)
print(f"\nSaved {len(df_raw):,} raw rows â†’ {RAW_FILE}")


In [ ]:
# â”€â”€ Stage 2b: Parse â€” split transcripts into prepared_remarks / qa_text â”€â”€â”€â”€â”€â”€
# Reads from data/transcripts_raw.parquet (written by Stage 2a).
# Safe to re-run without re-downloading.

import json, os
import pandas as pd

RAW_FILE = "data/transcripts_raw.parquet"
OUT_FILE = "data/transcripts_filtered.parquet"
KEEP_COLS = ["symbol", "quarter", "year", "date",
             "prepared_remarks", "qa_text", "word_count"]

df_raw = pd.read_parquet(RAW_FILE)
print(f"Loaded {len(df_raw):,} rows from {RAW_FILE}")

def parse_transcript(row):
    sc = row.get("structured_content")
    if sc is None:
        sc = []
    elif isinstance(sc, str):
        try:    sc = json.loads(sc)
        except: sc = []
    else:
        sc = list(sc)
    if len(sc) == 0:
        return "", "", 0

    seen = []
    for turn in sc:
        spk = (turn.get("speaker") or "").strip()
        if spk and spk not in seen:
            seen.append(spk)
        if len(seen) == 5:
            break
    mgmt = set(seen)

    split_idx = len(sc)
    for i, turn in enumerate(sc):
        if (turn.get("speaker") or "").strip() not in mgmt:
            split_idx = i
            break

    prepared = " ".join((t.get("text") or "") for t in sc[:split_idx]).strip()
    qa       = " ".join((t.get("text") or "") for t in sc[split_idx:]).strip()
    return prepared, qa, len(prepared.split()) + len(qa.split())

print("Parsing transcripts...")
parsed         = df_raw.apply(parse_transcript, axis=1, result_type="expand")
parsed.columns = ["prepared_remarks", "qa_text", "word_count"]
out            = pd.concat([df_raw.drop(columns=["structured_content"], errors="ignore"),
                             parsed], axis=1)[KEEP_COLS]
out["date"]    = pd.to_datetime(out["date"], errors="coerce")

out.to_parquet(OUT_FILE, index=False)
print(f"Saved {len(out):,} rows â†’ {OUT_FILE}")

per_ticker = out.groupby("symbol").size().sort_values(ascending=False)
print(f"\n{'='*55}")
print(f"Total rows: {len(out):,}")
print(f"\nRows per ticker:\n{per_ticker.to_string()}")
print(f"\nRows with non-empty qa_text: {(out['qa_text'].str.len() > 0).sum():,}")

thin = per_ticker[per_ticker < 20]
print("\nTickers with < 20 quarters:" if not thin.empty else "\nAll tickers have >= 20 quarters.")
if not thin.empty:
    print(thin.to_string())

print(f"\nDate range per ticker:")
dr = out.groupby("symbol")["date"].agg(Min="min", Max="max")
dr["Min"] = dr["Min"].dt.date
dr["Max"] = dr["Max"].dt.date
print(dr.sort_index().to_string())


### Transcript pre-processing

In [ ]:
# â”€â”€ Step 3: Preprocess transcript text â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import re
import pandas as pd

IN_FILE  = "data/transcripts_filtered.parquet"
OUT_FILE = "data/transcripts_preprocessed.parquet"

df = pd.read_parquet(IN_FILE)
print(f"Loaded {len(df):,} rows")

# â”€â”€ Cleaning patterns (compiled once) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Sentences to drop entirely (matched case-insensitively)
OPERATOR_PHRASES = re.compile(
    r"your next question|please go ahead|the floor is yours|"
    r"lines are open|operator instructions",
    re.IGNORECASE,
)
BOILERPLATE_PHRASES = re.compile(
    r"safe harbor|forward.looking statements|cautionary|"
    r"sec filings|no obligation to update",
    re.IGNORECASE,
)
# Filler sentences: entire sentence is just a greeting/pleasantry
FILLER_SENTENCE = re.compile(
    r"^\s*(good\s+(morning|afternoon|evening)|thank\s+you|thanks\s+for\s+joining|welcome)\W*$",
    re.IGNORECASE,
)
# Inline phrase removal
SLIDE_REF = re.compile(r"turning\s+to\s+slide\s+\d+|slide\s+\d+", re.IGNORECASE)


def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    # Split into sentences on . ! ? followed by whitespace or end
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    cleaned = []
    for sent in sentences:
        # 1. Drop operator turns
        if OPERATOR_PHRASES.search(sent):
            continue
        # 2. Drop pure filler sentences
        if FILLER_SENTENCE.match(sent):
            continue
        # 3. Drop legal boilerplate sentences
        if BOILERPLATE_PHRASES.search(sent):
            continue
        # 4. Remove slide references inline (keep rest of sentence)
        sent = SLIDE_REF.sub("", sent)
        # 5. Normalise whitespace
        sent = re.sub(r"\s+", " ", sent).strip()
        if sent:
            cleaned.append(sent)

    return " ".join(cleaned)


# â”€â”€ Apply cleaning â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Cleaning prepared_remarks...")
df["prepared_remarks"] = df["prepared_remarks"].apply(clean_text)

print("Cleaning qa_text...")
df["qa_text"] = df["qa_text"].apply(clean_text)

# â”€â”€ Derived columns â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df["prep_word_count"] = df["prepared_remarks"].str.split().str.len().fillna(0).astype(int)
df["qa_word_count"]   = df["qa_text"].str.split().str.len().fillna(0).astype(int)
df["has_qa"]          = df["qa_word_count"] > 50
df["low_quality"]     = df["prep_word_count"] < 200

# â”€â”€ Save â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df.to_parquet(OUT_FILE, index=False)
print(f"Saved â†’ {OUT_FILE}\n")

# â”€â”€ Summary stats â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"{'='*55}")
print(f"Total rows       : {len(df):,}")
print(f"has_qa = True    : {df['has_qa'].sum():,}")
print(f"low_quality      : {df['low_quality'].sum():,}")

print(f"\nPer-ticker stats (mean word counts + quarter count):")
stats = df.groupby("symbol").agg(
    quarters        = ("quarter", "count"),
    mean_prep_words = ("prep_word_count", "mean"),
    mean_qa_words   = ("qa_word_count",   "mean"),
).round(0).astype({"quarters": int, "mean_prep_words": int, "mean_qa_words": int})
stats = stats.sort_index()
print(stats.to_string())

thin = stats[stats["quarters"] < 20]
if thin.empty:
    print("\nAll tickers have >= 20 quarters.")
else:
    print(f"\nTickers with fewer than 20 quarters:\n{thin[['quarters']].to_string()}")


### FinBERT sentiment scoring

In [12]:

# Quick fix directly in kernel â€” patch the pipeline and score_text then re-run scoring
import torch
from transformers import pipeline
import time
import pandas as pd

device = 0 if torch.cuda.is_available() else -1

print("Reloading FinBERT with top_k=None...")
finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    truncation=True,
    max_length=512,
    device=device,
    top_k=None,
)

CHUNK_WORDS = 450

def chunk_text(text, n=CHUNK_WORDS):
    words = text.split()
    return [" ".join(words[i:i+n]) for i in range(0, len(words), n)] if words else []

def score_text(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    chunks = chunk_text(text)
    if not chunks:
        return 0.0
    scores, weights = [], []
    for chunk in chunks:
        probs  = {r["label"].lower(): r["score"] for r in finbert(chunk)[0]}
        signed = probs.get("positive", 0.0) - probs.get("negative", 0.0)
        scores.append(signed)
        weights.append(len(chunk.split()))
    total_w = sum(weights)
    return sum(s * w for s, w in zip(scores, weights)) / total_w if total_w else 0.0

# Quick sanity check on one chunk
test = finbert("Revenue exceeded expectations this quarter.")[0]
print("Test output:", test)
print("FinBERT ready.")


Reloading FinBERT with top_k=None...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Test output: [{'label': 'positive', 'score': 0.9527550339698792}, {'label': 'neutral', 'score': 0.02446388266980648}, {'label': 'negative', 'score': 0.022781046107411385}]
FinBERT ready.


In [13]:

df = pd.read_parquet("data/transcripts_preprocessed.parquet")
prep_scores, qa_scores = [], []
t0 = time.time()

for i, row in df.iterrows():
    p = score_text(row["prepared_remarks"])
    q = score_text(row["qa_text"])
    prep_scores.append(p)
    qa_scores.append(q)

    n = len(prep_scores)
    if n % 25 == 0 or n == len(df):
        elapsed = time.time() - t0
        rate    = n / elapsed if elapsed > 0 else 0
        eta     = (len(df) - n) / rate / 60 if rate > 0 else 0
        print(f"[{n:>3}/{len(df)}]  {row['symbol']:5s}  Q{row['quarter']} {row['year']}  "
              f"prep={p:+.3f}  qa={q:+.3f}  ({elapsed:.0f}s, ~{eta:.0f} min left)")

df["prep_sentiment"] = prep_scores
df["qa_sentiment"]   = qa_scores
df.to_parquet("data/transcripts_scored.parquet", index=False)

total = time.time() - t0
print(f"\nSaved -> data/transcripts_scored.parquet")
print(f"Total: {total/60:.1f} min  ({total/len(df):.1f}s per row)")


[ 25/524]  CRM    Q4 2024  prep=+0.398  qa=+0.141  (446s, ~148 min left)
[ 50/524]  GOOGL  Q3 2020  prep=+0.435  qa=+0.206  (812s, ~128 min left)
[ 75/524]  GS     Q2 2021  prep=+0.701  qa=+0.255  (1156s, ~115 min left)
[100/524]  JPM    Q1 2022  prep=-0.138  qa=+0.051  (1638s, ~116 min left)
[125/524]  GS     Q4 2018  prep=+0.316  qa=+0.156  (2020s, ~107 min left)
[150/524]  AMD    Q3 2020  prep=-0.138  qa=+0.371  (2478s, ~103 min left)
[175/524]  AMZN   Q2 2021  prep=+0.531  qa=+0.309  (2822s, ~94 min left)
[200/524]  BKNG   Q1 2022  prep=+0.224  qa=+0.124  (3193s, ~86 min left)
[225/524]  CAT    Q4 2024  prep=-0.409  qa=-0.029  (3545s, ~79 min left)
[250/524]  UNH    Q3 2020  prep=+0.664  qa=+0.156  (4233s, ~77 min left)
[275/524]  XOM    Q2 2021  prep=+0.552  qa=+0.229  (5006s, ~76 min left)
[300/524]  AMD    Q1 2018  prep=-0.029  qa=+0.127  (5407s, ~67 min left)
[325/524]  MCD    Q4 2020  prep=+0.466  qa=+0.346  (5771s, ~59 min left)
[350/524]  META   Q3 2021  prep=+0.175  qa=+0.1

## Feature Generation
#### prep_sentiment, qa_sentiment, sentiment_surprise, sentiment_velocity, qa_vs_prep_divergence

In [14]:

import pandas as pd

df = pd.read_parquet("data/transcripts_scored.parquet")
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

df["sentiment_surprise"] = df.groupby("symbol")["prep_sentiment"].transform(
    lambda s: s - s.shift(1).expanding().mean()
)
df["sentiment_velocity"] = df.groupby("symbol")["prep_sentiment"].transform(
    lambda s: s - s.shift(1)
)
df["qa_vs_prep_divergence"] = df["prep_sentiment"] - df["qa_sentiment"]

KEEP = ["symbol", "quarter", "year", "date",
        "prep_sentiment", "qa_sentiment",
        "sentiment_surprise", "sentiment_velocity", "qa_vs_prep_divergence"]

df[KEEP].to_parquet("data/transcripts_features.parquet", index=False)
print("Saved -> data/transcripts_features.parquet")

FEATURES = ["prep_sentiment", "qa_sentiment",
            "sentiment_surprise", "sentiment_velocity", "qa_vs_prep_divergence"]

pd.set_option("display.float_format", "{:+.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

crm = df[df["symbol"] == "CRM"][["symbol","quarter","year","date"] + FEATURES].head(10)
print(f"\nCRM (first 10 rows):\n{crm.to_string(index=False)}")

print(f"\n{'='*60}")
print("Feature summary stats:")
stats = df[FEATURES].agg(["mean","std","min","max"]).T
stats["nan_count"] = df[FEATURES].isna().sum()
stats.columns = ["mean","std","min","max","nan_count"]
print(stats.to_string(float_format="{:+.4f}".format))

print(f"\nCorrelation matrix:")
print(df[FEATURES].corr().to_string(float_format="{:+.4f}".format))


Saved -> data/transcripts_features.parquet

CRM (first 10 rows):
symbol  quarter  year                date  prep_sentiment  qa_sentiment  sentiment_surprise  sentiment_velocity  qa_vs_prep_divergence
   CRM        1  2018 2017-05-18 10:45:00         +0.8769       +0.1603                 NaN                 NaN                +0.7166
   CRM        2  2018 2017-08-22 17:00:00         +0.4065       +0.3385             -0.4703             -0.4703                +0.0681
   CRM        3  2018 2017-11-21 00:00:00         +0.6798       +0.1391             +0.0381             +0.2732                +0.5407
   CRM        4  2018 2018-02-28 17:00:00         +0.4528       +0.3291             -0.2016             -0.2270                +0.1237
   CRM        1  2019 2018-05-29 17:00:00         +0.4240       +0.2824             -0.1801             -0.0289                +0.1416
   CRM        2  2019 2018-08-29 17:00:00         +0.6657       +0.2507             +0.0978             +0.2418              

In [ ]:
# â”€â”€ Step 6a: Download OHLCV + compute forward returns â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys
import pandas as pd
import numpy as np

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

try:
    import yfinance as yf
except ImportError:
    pip_install("yfinance")
    import yfinance as yf

features  = pd.read_parquet("data/transcripts_features.parquet")
symbols   = sorted(features["symbol"].unique().tolist())
HORIZONS  = [5, 10, 21, 42, 63]

# Date range: one year before earliest transcript, one year after latest
start = (pd.to_datetime(features["date"].min()) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
end   = (pd.to_datetime(features["date"].max()) + pd.DateOffset(years=1)).strftime("%Y-%m-%d")
print(f"Downloading {len(symbols)} tickers  {start} -> {end} ...")

raw   = yf.download(symbols, start=start, end=end, auto_adjust=True, progress=False)
close = raw["Close"].sort_index()          # DataFrame: date-index x ticker columns
print(f"Price data: {close.shape[0]} trading days x {close.shape[1]} tickers")

# â”€â”€ Compute forward returns â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
trading_days = close.index  # DatetimeIndex of all trading days in the window

records = []
for _, row in features.iterrows():
    sym  = row["symbol"]
    edate = pd.Timestamp(row["date"]).normalize()  # earnings date (transcript date)

    # Entry = first trading day strictly after earnings date
    future = trading_days[trading_days > edate]
    if len(future) == 0:
        continue
    entry_date  = future[0]
    entry_idx   = close.index.get_loc(entry_date)
    entry_price = close.at[entry_date, sym]

    if pd.isna(entry_price) or entry_price == 0:
        continue

    rec = {
        "symbol"        : sym,
        "quarter"       : row["quarter"],
        "year"          : row["year"],
        "earnings_date" : edate,
        "entry_date"    : entry_date,
    }

    for h in HORIZONS:
        exit_idx = entry_idx + h
        if exit_idx >= len(close):
            rec[f"fwd_{h}d"] = np.nan
        else:
            exit_price = close.iloc[exit_idx][sym]
            rec[f"fwd_{h}d"] = (exit_price / entry_price - 1) if exit_price > 0 else np.nan

    records.append(rec)

returns = pd.DataFrame(records)
returns.to_parquet("data/earnings_returns.parquet", index=False)
print(f"\nSaved {len(returns):,} rows -> data/earnings_returns.parquet")
print(f"Columns: {list(returns.columns)}")
print(f"\nSample (5 rows):")
print(returns.head().to_string(index=False))
print(f"\nForward return means:")
for h in HORIZONS:
    col = f"fwd_{h}d"
    print(f"  {col}: {returns[col].mean()*100:+.2f}%  (nan={returns[col].isna().sum()})")


In [ ]:
# â”€â”€ Step 6b: Merge + Spearman IC analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

features = pd.read_parquet("data/transcripts_features.parquet")
returns  = pd.read_parquet("data/earnings_returns.parquet")

features["earnings_date"] = pd.to_datetime(features["date"]).dt.normalize()
returns["earnings_date"]  = pd.to_datetime(returns["earnings_date"])

# â”€â”€ Merge on symbol + nearest earnings_date within 7-day window â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
features = features.sort_values("earnings_date")
returns  = returns.sort_values("earnings_date")

print(f"Before merge: features={len(features):,}  returns={len(returns):,}")

merged = pd.merge_asof(
    features,
    returns[["symbol","earnings_date","entry_date"] +
            [f"fwd_{h}d" for h in [5,10,21,42,63]]],
    on="earnings_date",
    by="symbol",
    tolerance=pd.Timedelta("7 days"),
    direction="nearest",
)
merged = merged.dropna(subset=["entry_date"])
print(f"After merge  : {len(merged):,} rows")

merged.to_parquet("data/master_dataset.parquet", index=False)
print("Saved -> data/master_dataset.parquet\n")

# â”€â”€ Spearman IC â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
FEATURES  = ["prep_sentiment","qa_sentiment","sentiment_surprise",
             "sentiment_velocity","qa_vs_prep_divergence"]
HORIZONS  = [5, 10, 21, 42, 63]
TICKERS   = sorted(merged["symbol"].unique())

def spearman_ic(x, y):
    mask = ~(x.isna() | y.isna())
    if mask.sum() < 5:
        return np.nan, np.nan, np.nan
    r, p = stats.spearmanr(x[mask], y[mask])
    n  = mask.sum()
    t  = r * np.sqrt((n - 2) / (1 - r**2 + 1e-12))
    return r, t, p

# Per-ticker IC then average across tickers
rows = []
for feat in FEATURES:
    row = {"feature": feat}
    for h in HORIZONS:
        col     = f"fwd_{h}d"
        ticker_ics = []
        for tkr in TICKERS:
            sub = merged[merged["symbol"] == tkr]
            ic, _, _ = spearman_ic(sub[feat], sub[col])
            if not np.isnan(ic):
                ticker_ics.append(ic)
        if not ticker_ics:
            row[f"ic_{h}d"] = np.nan
            row[f"t_{h}d"]  = np.nan
            row[f"p_{h}d"]  = np.nan
            continue
        mean_ic = np.mean(ticker_ics)
        n   = len(ticker_ics)
        se  = np.std(ticker_ics, ddof=1) / np.sqrt(n) + 1e-12
        t   = mean_ic / se
        p   = 2 * stats.t.sf(abs(t), df=n-1)
        row[f"ic_{h}d"] = mean_ic
        row[f"t_{h}d"]  = t
        row[f"p_{h}d"]  = p
    rows.append(row)

ic_df = pd.DataFrame(rows).set_index("feature")

# â”€â”€ Print IC table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Table 1 â€” Mean Spearman IC across tickers")
print(f"{'Feature':<26}", end="")
for h in HORIZONS:
    print(f"{'|':>2} {h:>2}d    ", end="")
print()
print("-" * 70)
for feat in FEATURES:
    print(f"{feat:<26}", end="")
    for h in HORIZONS:
        ic = ic_df.at[feat, f"ic_{h}d"]
        p  = ic_df.at[feat, f"p_{h}d"]
        sig = "*" if (not np.isnan(p) and p < 0.05) else " "
        print(f"  {ic:+.3f}{sig} ", end="")
    print()
print("\n* p < 0.05")

# â”€â”€ Plot IC decay curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
os.makedirs("outputs", exist_ok=True)
COLORS = ["#2196F3","#FF9800","#4CAF50","#9C27B0","#F44336"]
LABELS = {
    "prep_sentiment"       : "Prep Sentiment",
    "qa_sentiment"         : "QA Sentiment",
    "sentiment_surprise"   : "Sentiment Surprise",
    "sentiment_velocity"   : "Sentiment Velocity",
    "qa_vs_prep_divergence": "QA vs Prep Divergence",
}

fig, ax = plt.subplots(figsize=(10, 5))
for feat, color in zip(FEATURES, COLORS):
    ics = [ic_df.at[feat, f"ic_{h}d"] for h in HORIZONS]
    ax.plot(HORIZONS, ics, marker="o", label=LABELS[feat],
            color=color, linewidth=2, markersize=6)

ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Forward Return Horizon (trading days)")
ax.set_ylabel("Mean Spearman IC")
ax.set_title("IC Decay Curves â€” Sentiment Features vs Forward Returns")
ax.set_xticks(HORIZONS)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/ic_decay_curves.png", dpi=150)
plt.close()
print("\nPlot saved -> outputs/ic_decay_curves.png")


In [ ]:

# ── Step 7: Backtest — sentiment_velocity long/short strategy ─────────────────
import pandas as pd, numpy as np, os
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

# ── Parameters ────────────────────────────────────────────────────────────────
CAPITAL   = 100_000
POS_SIZE  = 0.10
MAX_POS   = 10
HOLD_DAYS = 21
TC_RT     = 0.0010        # 10 bps round trip
TC_HALF   = TC_RT / 2
RF_ANN    = 0.04

# ── Load signals ──────────────────────────────────────────────────────────────
master = pd.read_parquet("data/master_dataset.parquet")
master = master.dropna(subset=["sentiment_velocity"])
master = master[master["sentiment_velocity"] != 0].copy()
master["entry_date"] = pd.to_datetime(master["entry_date"])
master["earnings_date"] = pd.to_datetime(master["earnings_date"])

symbols = sorted(master["symbol"].unique().tolist())

# ── Download OHLCV + SPY ──────────────────────────────────────────────────────
all_tix  = symbols + ["SPY"]
d_min    = master["entry_date"].min() - pd.DateOffset(days=5)
d_max    = master["entry_date"].max() + pd.DateOffset(days=HOLD_DAYS + 30)
print(f"Downloading {len(all_tix)} tickers  {d_min.date()} → {d_max.date()} ...")
raw    = yf.download(all_tix, start=d_min.strftime("%Y-%m-%d"),
                     end=d_max.strftime("%Y-%m-%d"), auto_adjust=True, progress=False)
opens  = raw["Open"].ffill()
closes = raw["Close"].ffill()
tdates = closes.index
print(f"Price data: {len(tdates)} trading days  tickers={closes.shape[1]}")

# ── Build event table with exact entry/exit dates ─────────────────────────────
events = []
for _, row in master.iterrows():
    vel  = row["sentiment_velocity"]
    sym  = row["symbol"]
    eday = row["entry_date"].normalize()

    if eday not in tdates:
        continue
    eidx = tdates.get_loc(eday)
    xidx = eidx + HOLD_DAYS
    if xidx >= len(tdates):
        continue

    xday   = tdates[xidx]
    o_px   = opens.at[eday, sym]
    x_px   = closes.at[xday, sym]
    if pd.isna(o_px) or pd.isna(x_px) or o_px <= 0:
        continue

    events.append(dict(
        symbol=sym, entry_date=eday, exit_date=xday,
        direction=1 if vel > 0 else -1,
        abs_vel=abs(vel), entry_open=o_px,
    ))

ev = pd.DataFrame(events).sort_values("entry_date").reset_index(drop=True)
print(f"\nSignals: {len(ev)}  long={( ev.direction==1).sum()}  short={(ev.direction==-1).sum()}")

# ── Simulation ────────────────────────────────────────────────────────────────
cash     = float(CAPITAL)
open_pos = []        # list of active position dicts
daily    = []        # {date, portfolio}
trades   = []        # completed trade records

sim_start = ev["entry_date"].min()
sim_end   = ev["exit_date"].max()
sim_days  = tdates[(tdates >= sim_start) & (tdates <= sim_end)]

prev_val  = float(CAPITAL)

for day in sim_days:
    # 1. Process exits (at today's close)
    alive = []
    for pos in open_pos:
        if pos["exit_date"] == day:
            c = closes.at[day, pos["symbol"]]
            if pd.isna(c): c = pos["entry_open"]
            pnl    = pos["direction"] * pos["shares"] * (c - pos["entry_open"])
            tc_ex  = abs(pos["shares"] * c) * TC_HALF
            cash  += pos["allocated"] + pnl - tc_ex
            raw_r  = pos["direction"] * (c / pos["entry_open"] - 1)
            trades.append(dict(
                symbol=pos["symbol"], direction=pos["direction"],
                entry_date=pos["entry_date"], exit_date=day,
                raw_return=raw_r, net_return=raw_r - TC_RT,
            ))
        else:
            alive.append(pos)
    open_pos = alive

    # 2. Process entries (at today's open), sized on prev day portfolio value
    if prev_val > 0:
        sigs = ev[ev["entry_date"] == day].sort_values("abs_vel", ascending=False)
        for _, sig in sigs.iterrows():
            if len(open_pos) >= MAX_POS:
                break
            o = opens.at[day, sig["symbol"]]
            if pd.isna(o) or o <= 0:
                continue
            allocated = POS_SIZE * prev_val
            tc_en     = allocated * TC_HALF
            shares    = allocated / o
            cash     -= allocated + tc_en
            open_pos.append(dict(
                symbol=sig["symbol"], entry_date=day, exit_date=sig["exit_date"],
                direction=sig["direction"], entry_open=o,
                shares=shares, allocated=allocated,
            ))

    # 3. Mark to market at today's close
    pos_val = 0.0
    for pos in open_pos:
        c = closes.at[day, pos["symbol"]]
        if pd.isna(c): c = pos["entry_open"]
        pos_val += pos["allocated"] + pos["direction"] * pos["shares"] * (c - pos["entry_open"])

    port_val = cash + pos_val
    daily.append({"date": day, "portfolio": port_val})
    prev_val = port_val

daily_df = pd.DataFrame(daily).set_index("date")
trade_df = pd.DataFrame(trades)

# SPY benchmark normalised to CAPITAL on sim_start
spy = closes["SPY"].loc[sim_start:sim_end]
spy_normed = spy / spy.iloc[0] * CAPITAL

# ── Performance metrics ───────────────────────────────────────────────────────
pv     = daily_df["portfolio"]
rets   = pv.pct_change().dropna()
spy_r  = spy_normed.pct_change().dropna()

n_yrs  = len(pv) / 252
cagr   = (pv.iloc[-1] / CAPITAL) ** (1 / n_yrs) - 1

rf_d   = (1 + RF_ANN) ** (1/252) - 1
excess = rets - rf_d
sharpe = excess.mean() / rets.std() * np.sqrt(252)

neg_ex = rets[rets < rf_d] - rf_d
sortino = excess.mean() / neg_ex.std() * np.sqrt(252) if len(neg_ex) > 0 else np.nan

roll_max  = pv.cummax()
dd        = (pv - roll_max) / roll_max
max_dd    = dd.min()

# Drawdown duration (calendar days)
in_dd, t0_dd, dd_durs = False, None, []
for d, v in dd.items():
    if v < 0 and not in_dd:
        in_dd, t0_dd = True, d
    elif v >= 0 and in_dd:
        dd_durs.append((d - t0_dd).days); in_dd = False
if in_dd and t0_dd:
    dd_durs.append((dd.index[-1] - t0_dd).days)
max_dd_dur = max(dd_durs) if dd_durs else 0

calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan

longs  = trade_df[trade_df["direction"] ==  1]
shorts = trade_df[trade_df["direction"] == -1]
win_r  = (trade_df["net_return"] > 0).mean()
l_wr   = (longs["net_return"]  > 0).mean() if len(longs)  else np.nan
s_wr   = (shorts["net_return"] > 0).mean() if len(shorts) else np.nan
avg_r  = trade_df["net_return"].mean()

common = rets.index.intersection(spy_r.index)
beta, alpha_d, _, _, _ = stats.linregress(spy_r.loc[common], rets.loc[common])
alpha_ann = alpha_d * 252

spy_cagr = (spy_normed.iloc[-1] / CAPITAL) ** (1 / n_yrs) - 1

# ── Print results ─────────────────────────────────────────────────────────────
W = 32
print("=" * 52)
print("  SENTIMENT VELOCITY BACKTEST  (2018–2024)")
print("=" * 52)
print(f"\n{'Return metrics':}")
print(f"  {'CAGR':<{W}} {cagr*100:>+8.2f}%   (SPY: {spy_cagr*100:>+.2f}%)")
print(f"  {'Sharpe  (rf=4%)':<{W}} {sharpe:>9.3f}")
print(f"  {'Sortino (rf=4%)':<{W}} {sortino:>9.3f}")
print(f"  {'Calmar':<{W}} {calmar:>9.3f}")
print(f"\n{'Drawdown':}")
print(f"  {'Max Drawdown':<{W}} {max_dd*100:>+8.2f}%")
print(f"  {'Max DD Duration':<{W}} {max_dd_dur:>8} days")
print(f"\n{'Trade stats':}")
print(f"  {'Total trades':<{W}} {len(trade_df):>8}")
print(f"  {'Win rate (all)':<{W}} {win_r*100:>8.1f}%")
print(f"  {'Win rate (long)':<{W}} {l_wr*100:>8.1f}%")
print(f"  {'Win rate (short)':<{W}} {s_wr*100:>8.1f}%")
print(f"  {'Avg net return / trade':<{W}} {avg_r*100:>+8.3f}%")
print(f"\n{'Market exposure':}")
print(f"  {'Beta vs SPY':<{W}} {beta:>9.3f}")
print(f"  {'Alpha vs SPY (annualised)':<{W}} {alpha_ann*100:>+8.2f}%")

# ── Conclusion ────────────────────────────────────────────────────────────────
print("\n" + "=" * 52)
print("CONCLUSION")
print("=" * 52)
if alpha_ann > 0.03 and sharpe > 0.8:
    verdict = (
        f"YES — sentiment velocity generates alpha over buy-and-hold SPY. "
        f"Strategy CAGR {cagr*100:.1f}% vs SPY {spy_cagr*100:.1f}%, "
        f"Sharpe {sharpe:.2f}, and {alpha_ann*100:.1f}% annualised alpha. "
        f"The win rate advantage on the long side ({l_wr*100:.0f}%) confirms "
        f"the market underreacts to improving management tone post-earnings."
    )
elif alpha_ann > 0:
    verdict = (
        f"MARGINAL — sentiment velocity shows {alpha_ann*100:.1f}% annualised alpha over SPY "
        f"but with Sharpe {sharpe:.2f} the risk-adjusted case is weak in isolation. "
        f"The short leg likely drags performance in a sustained bull market (2018–2024). "
        f"Consider long-only or pairing with a macro regime filter."
    )
else:
    verdict = (
        f"NO — sentiment velocity does not beat buy-and-hold SPY in this backtest. "
        f"Strategy CAGR {cagr*100:.1f}% vs SPY {spy_cagr*100:.1f}%. "
        f"The short leg loses against a 7-year bull trend. "
        f"The IC signal at 21d ({'+0.106'}) is real but the short side costs more than it earns."
    )
print(verdict)

# ── Plots ─────────────────────────────────────────────────────────────────────
os.makedirs("outputs", exist_ok=True)

# 1. Equity curve
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_df.index, daily_df["portfolio"],
        label="Sentiment Velocity Strategy", color="#2196F3", linewidth=1.5)
ax.plot(spy_normed.index, spy_normed,
        label=f"SPY buy & hold  (CAGR {spy_cagr*100:.1f}%)",
        color="#FF9800", linewidth=1.5, linestyle="--")
ax.axhline(CAPITAL, color="gray", linewidth=0.7, linestyle=":")
ax.set_ylabel("Portfolio Value ($)")
ax.set_title(f"Equity Curve — Sentiment Velocity Strategy vs SPY\nCAGR {cagr*100:.1f}%  Sharpe {sharpe:.2f}  Max DD {max_dd*100:.1f}%")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/sentiment_velocity_equity.png", dpi=150)
plt.close()
print("\nSaved outputs/sentiment_velocity_equity.png")

# 2. Drawdown
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(dd.index, dd * 100, 0, alpha=0.45, color="#F44336", label=f"Max DD {max_dd*100:.1f}%")
ax.plot(dd.index, dd * 100, color="#F44336", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Drawdown (%)")
ax.set_title("Strategy Drawdown")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/sentiment_velocity_drawdown.png", dpi=150)
plt.close()
print("Saved outputs/sentiment_velocity_drawdown.png")

# 3. Trade return distribution
fig, ax = plt.subplots(figsize=(10, 5))
all_r = trade_df["net_return"] * 100
bins  = np.linspace(all_r.quantile(0.02), all_r.quantile(0.98), 40)
ax.hist(longs["net_return"]  * 100, bins=bins, alpha=0.65, color="#2196F3",
        label=f"Long  n={len(longs)}  mean={longs['net_return'].mean()*100:+.2f}%")
ax.hist(shorts["net_return"] * 100, bins=bins, alpha=0.65, color="#F44336",
        label=f"Short n={len(shorts)}  mean={shorts['net_return'].mean()*100:+.2f}%")
ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_xlabel("Net Return per Trade (%)")
ax.set_ylabel("Count")
ax.set_title("Trade Return Distribution — Long vs Short")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/sentiment_velocity_trades.png", dpi=150)
plt.close()
print("Saved outputs/sentiment_velocity_trades.png")


In [ ]:

# Step 8: Long-only backtest — fixed 10% sizing, 21d hold
import pandas as pd, numpy as np, os
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

# Parameters
CAPITAL   = 100_000
HOLD_DAYS = 21
TC_RT     = 0.0010
TC_HALF   = TC_RT / 2
RF_ANN    = 0.04

MAX_POS = 10
POS_SIZE = 0.10
COLOR = "#4CAF50"

# Load long signals only
master = pd.read_parquet("data/master_dataset.parquet")
master = master.dropna(subset=["sentiment_velocity"])
master = master[master["sentiment_velocity"] > 0].copy()
master["entry_date"] = pd.to_datetime(master["entry_date"])
symbols = sorted(master["symbol"].unique().tolist())

# Download prices
all_tix = symbols + ["SPY"]
d_min   = master["entry_date"].min() - pd.DateOffset(days=5)
d_max   = master["entry_date"].max() + pd.DateOffset(days=HOLD_DAYS + 30)
print(f"Downloading {d_min.date()} to {d_max.date()} ...")
raw    = yf.download(all_tix, start=d_min.strftime("%Y-%m-%d"),
                     end=d_max.strftime("%Y-%m-%d"), auto_adjust=True, progress=False)
opens  = raw["Open"].ffill()
closes = raw["Close"].ffill()
tdates = closes.index
print(f"{len(tdates)} trading days\n")

# Build event table
events = []
for _, row in master.iterrows():
    sym  = row["symbol"]
    eday = row["entry_date"].normalize()
    if eday not in tdates: continue
    eidx = tdates.get_loc(eday)
    xidx = eidx + HOLD_DAYS
    if xidx >= len(tdates): continue
    xday = tdates[xidx]
    o = opens.at[eday, sym]; x = closes.at[xday, sym]
    if pd.isna(o) or pd.isna(x) or o <= 0: continue
    events.append(dict(symbol=sym, entry_date=eday, exit_date=xday,
                       abs_vel=abs(row["sentiment_velocity"]), entry_open=o))

ev = pd.DataFrame(events).sort_values("entry_date").reset_index(drop=True)
print(f"Long signals: {len(ev)}")

# Simulation
cash = float(CAPITAL); open_pos = []; daily = []; trades = []
prev_val = float(CAPITAL)
sim_days = tdates[(tdates >= ev["entry_date"].min()) & (tdates <= ev["exit_date"].max())]

for day in sim_days:
    # Exits at scheduled date
    alive = []
    for pos in open_pos:
        c = closes.at[day, pos["symbol"]]
        if pd.isna(c): c = pos["entry_open"]
        if pos["exit_date"] == day:
            pnl   = pos["shares"] * (c - pos["entry_open"])
            tc_ex = abs(pos["shares"] * c) * TC_HALF
            cash += pos["allocated"] + pnl - tc_ex
            trades.append(dict(net_return=c / pos["entry_open"] - 1 - TC_RT))
        else:
            alive.append(pos)
    open_pos = alive

    # New entries at open, prioritised by abs velocity
    for _, sig in ev[ev["entry_date"] == day].sort_values("abs_vel", ascending=False).iterrows():
        if len(open_pos) >= MAX_POS: break
        o = opens.at[day, sig["symbol"]]
        if pd.isna(o) or o <= 0: continue
        alloc = min(POS_SIZE * prev_val, max(cash, 0))
        if alloc <= 0: continue
        cash -= alloc + alloc * TC_HALF
        open_pos.append(dict(symbol=sig["symbol"], entry_date=day,
                             exit_date=sig["exit_date"], entry_open=o,
                             shares=alloc / o, allocated=alloc))

    # MTM at close
    pos_val = sum(
        pos["allocated"] + pos["shares"] * (closes.at[day, pos["symbol"]] - pos["entry_open"])
        if not pd.isna(closes.at[day, pos["symbol"]]) else pos["allocated"]
        for pos in open_pos
    )
    port_val = cash + pos_val
    daily.append({"date": day, "portfolio": port_val})
    prev_val = port_val

daily_df = pd.DataFrame(daily).set_index("date")
trade_df = pd.DataFrame(trades)

spy      = closes["SPY"].loc[daily_df.index[0]:daily_df.index[-1]]
spy_n    = spy / spy.iloc[0] * CAPITAL
spy_r    = spy_n.pct_change().dropna()

# Metrics
pv      = daily_df["portfolio"]
rets    = pv.pct_change().dropna()
n_yrs   = len(pv) / 252
cagr    = (pv.iloc[-1] / CAPITAL) ** (1/n_yrs) - 1
rf_d    = (1 + RF_ANN) ** (1/252) - 1
ex      = rets - rf_d
sharpe  = ex.mean() / rets.std() * np.sqrt(252)
neg_ex  = rets[rets < rf_d] - rf_d
sortino = ex.mean() / neg_ex.std() * np.sqrt(252)
roll_max = pv.cummax()
dd      = (pv - roll_max) / roll_max
max_dd  = dd.min()
in_dd, t0, durs = False, None, []
for d, v in dd.items():
    if v < 0 and not in_dd: in_dd, t0 = True, d
    elif v >= 0 and in_dd:  durs.append((d - t0).days); in_dd = False
if in_dd and t0: durs.append((dd.index[-1] - t0).days)
calmar    = cagr / abs(max_dd) if max_dd else 0
win_r     = (trade_df["net_return"] > 0).mean()
avg_r     = trade_df["net_return"].mean()
spy_cagr  = (spy_n.iloc[-1] / CAPITAL) ** (1/n_yrs) - 1
cm        = rets.index.intersection(spy_r.index)
beta, alpha_d, *_ = stats.linregress(spy_r.loc[cm], rets.loc[cm])
alpha_ann = alpha_d * 252

W = 30
print("=" * 52)
print("  LONG-ONLY SENTIMENT VELOCITY (21d)")
print("=" * 52)
print(f"  {'CAGR':<{W}} {cagr*100:>+.2f}%  (SPY: {spy_cagr*100:>+.2f}%)")
print(f"  {'Sharpe  (rf=4%)':<{W}} {sharpe:>+.3f}")
print(f"  {'Sortino (rf=4%)':<{W}} {sortino:>+.3f}")
print(f"  {'Calmar':<{W}} {calmar:>+.3f}")
print(f"  {'Max Drawdown':<{W}} {max_dd*100:>+.2f}%")
print(f"  {'Max DD Duration':<{W}} {max(durs) if durs else 0:>} days")
print(f"  {'Total Trades':<{W}} {len(trade_df):>}")
print(f"  {'Win Rate':<{W}} {win_r*100:>.1f}%")
print(f"  {'Avg Net Return / Trade':<{W}} {avg_r*100:>+.3f}%")
print(f"  {'Beta vs SPY':<{W}} {beta:>+.3f}")
print(f"  {'Alpha vs SPY (annualised)':<{W}} {alpha_ann*100:>+.2f}%")

os.makedirs("outputs", exist_ok=True)

# Equity curve
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily_df.index, daily_df["portfolio"],
        label=f"Long-Only Fixed 10%  CAGR {cagr*100:.1f}%  Sharpe {sharpe:.2f}",
        color=COLOR, linewidth=1.5)
ax.plot(spy_n.index, spy_n,
        label=f"SPY Buy & Hold  CAGR {spy_cagr*100:.1f}%",
        color="#FF9800", linewidth=1.5, linestyle="--")
ax.axhline(CAPITAL, color="gray", linewidth=0.7, linestyle=":")
ax.set_ylabel("Portfolio Value ($)")
ax.set_title(f"Long-Only Sentiment Velocity — 21d Hold\n"
             f"CAGR {cagr*100:.1f}%  Sharpe {sharpe:.2f}  Sortino {sortino:.2f}  Max DD {max_dd*100:.1f}%")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(True, alpha=0.3); fig.tight_layout()
fig.savefig("outputs/longonly_equity.png", dpi=150); plt.close()
print("\nSaved outputs/longonly_equity.png")

# Drawdown
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(dd.index, dd*100, 0, alpha=0.45, color=COLOR,
                label=f"Max DD {max_dd*100:.1f}%  Duration {max(durs) if durs else 0}d")
ax.plot(dd.index, dd*100, color=COLOR, linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylabel("Drawdown (%)")
ax.set_title("Strategy Drawdown — Long-Only 21d")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); fig.tight_layout()
fig.savefig("outputs/longonly_drawdown.png", dpi=150); plt.close()
print("Saved outputs/longonly_drawdown.png")

# Trade return distribution
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(trade_df["net_return"].quantile(0.02)*100,
                   trade_df["net_return"].quantile(0.98)*100, 40)
ax.hist(trade_df["net_return"]*100, bins=bins, alpha=0.75, color=COLOR,
        label=f"n={len(trade_df)}  mean={avg_r*100:+.2f}%")
ax.axvline(0, color="black", linewidth=1.1, linestyle="--")
ax.axvline(avg_r*100, color=COLOR, linewidth=1.8, linestyle="-.",
           label=f"Mean {avg_r*100:+.2f}%")
ax.set_xlabel("Net Return per Trade (%)")
ax.set_ylabel("Count")
ax.set_title(f"Trade Return Distribution  Win Rate {win_r*100:.0f}%")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); fig.tight_layout()
fig.savefig("outputs/longonly_trades.png", dpi=150); plt.close()
print("Saved outputs/longonly_trades.png")


### sentiment-velocity PEAD master backtest

In [31]:

# ═══════════════════════════════════════════════════════════
#  Step 9: Long-only backtest — tweak parameters here
# ═══════════════════════════════════════════════════════════

# ── PARAMETERS ──────────────────────────────────────────────
STARTING_CAPITAL = 100_000   # initial portfolio value ($)
HOLD_DAYS        = 60        # trading days to hold each position
POS_SIZE         = 0.15      # fraction of portfolio per trade (0.10 = 10%)
MAX_POSITIONS    = 10        # max simultaneous open positions
TC_BPS           = 10        # round-trip transaction cost in basis points
RF_ANNUAL        = 0.04      # risk-free rate for Sharpe / Sortino
STOP_LOSS        = None      # position stop-loss, e.g. -0.10 = -10% | None = off
MIN_VELOCITY     = 0.0       # min abs sentiment_velocity to trade (0 = all signals)
LONG_ONLY        = True      # True = long only | False = long + short
# ────────────────────────────────────────────────────────────

import pandas as pd, numpy as np, os
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

TC_HALF = (TC_BPS / 10_000) / 2

# Load signals
master = pd.read_parquet("data/master_dataset.parquet")
master = master.dropna(subset=["sentiment_velocity"])
master["entry_date"] = pd.to_datetime(master["entry_date"])

if LONG_ONLY:
    master = master[master["sentiment_velocity"] > MIN_VELOCITY].copy()
else:
    master = master[master["sentiment_velocity"].abs() > MIN_VELOCITY].copy()

symbols = sorted(master["symbol"].unique().tolist())
print(f"Signals after filter: {len(master)}  "
      f"({'long only' if LONG_ONLY else 'long + short'}  min_vel>{MIN_VELOCITY})")

# Download prices
all_tix = symbols + ["SPY"]
d_min   = master["entry_date"].min() - pd.DateOffset(days=5)
d_max   = master["entry_date"].max() + pd.DateOffset(days=HOLD_DAYS + 30)
print(f"Downloading prices {d_min.date()} to {d_max.date()} ...")
raw    = yf.download(all_tix, start=d_min.strftime("%Y-%m-%d"),
                     end=d_max.strftime("%Y-%m-%d"), auto_adjust=True, progress=False)
opens  = raw["Open"].ffill()
closes = raw["Close"].ffill()
tdates = closes.index
print(f"{len(tdates)} trading days\n")

# Build event table
events = []
for _, row in master.iterrows():
    sym  = row["symbol"]
    vel  = row["sentiment_velocity"]
    eday = row["entry_date"].normalize()
    if eday not in tdates: continue
    eidx = tdates.get_loc(eday)
    xidx = eidx + HOLD_DAYS
    if xidx >= len(tdates): continue
    xday = tdates[xidx]
    o    = opens.at[eday, sym]
    x    = closes.at[xday, sym]
    if pd.isna(o) or pd.isna(x) or o <= 0: continue
    events.append(dict(
        symbol=sym, entry_date=eday, exit_date=xday,
        direction=1 if vel > 0 else -1,
        abs_vel=abs(vel), entry_open=o,
    ))

ev = pd.DataFrame(events).sort_values("entry_date").reset_index(drop=True)
print(f"Tradeable events: {len(ev)}  "
      f"(long={( ev.direction==1).sum()}  short={(ev.direction==-1).sum()})")

# ── Simulation ────────────────────────────────────────────────
cash     = float(STARTING_CAPITAL)
open_pos = []
daily    = []
trades   = []
prev_val = float(STARTING_CAPITAL)

sim_days = tdates[(tdates >= ev["entry_date"].min()) & (tdates <= ev["exit_date"].max())]

for day in sim_days:
    # Exits + optional stop-loss (checked at close)
    alive = []
    for pos in open_pos:
        c = closes.at[day, pos["symbol"]]
        if pd.isna(c): c = pos["entry_open"]
        pos_ret  = pos["direction"] * (c / pos["entry_open"] - 1)
        hit_stop = STOP_LOSS is not None and pos_ret < STOP_LOSS
        if pos["exit_date"] == day or hit_stop:
            pnl    = pos["direction"] * pos["shares"] * (c - pos["entry_open"])
            tc_ex  = abs(pos["shares"] * c) * TC_HALF
            cash  += pos["allocated"] + pnl - tc_ex
            net_r  = pos["direction"] * (c / pos["entry_open"] - 1) - TC_BPS / 10_000
            trades.append(dict(
                symbol=pos["symbol"],
                entry_date=pos["entry_date"],
                exit_date=day,
                direction=pos["direction"],
                net_return=net_r,
                stopped=hit_stop,
            ))
        else:
            alive.append(pos)
    open_pos = alive

    # New entries at open, highest abs_vel first
    for _, sig in ev[ev["entry_date"] == day].sort_values("abs_vel", ascending=False).iterrows():
        if len(open_pos) >= MAX_POSITIONS: break
        o = opens.at[day, sig["symbol"]]
        if pd.isna(o) or o <= 0: continue
        alloc = min(POS_SIZE * prev_val, max(cash, 0))
        if alloc <= 0: continue
        cash -= alloc + alloc * TC_HALF
        open_pos.append(dict(
            symbol=sig["symbol"],
            entry_date=day,
            exit_date=sig["exit_date"],
            direction=sig["direction"],
            entry_open=o,
            shares=alloc / o,
            allocated=alloc,
        ))

    # Mark to market at close
    pos_val = 0.0
    for pos in open_pos:
        c = closes.at[day, pos["symbol"]]
        if pd.isna(c): c = pos["entry_open"]
        pos_val += pos["allocated"] + pos["direction"] * pos["shares"] * (c - pos["entry_open"])

    port_val = cash + pos_val
    daily.append({"date": day, "portfolio": port_val})
    prev_val = port_val

daily_df = pd.DataFrame(daily).set_index("date")
trade_df = pd.DataFrame(trades)

spy      = closes["SPY"].loc[daily_df.index[0]:daily_df.index[-1]]
spy_n    = spy / spy.iloc[0] * STARTING_CAPITAL
spy_r    = spy_n.pct_change().dropna()

# ── Metrics ───────────────────────────────────────────────────
pv       = daily_df["portfolio"]
rets     = pv.pct_change().dropna()
n_yrs    = len(pv) / 252
cagr     = (pv.iloc[-1] / STARTING_CAPITAL) ** (1 / n_yrs) - 1
rf_d     = (1 + RF_ANNUAL) ** (1 / 252) - 1
ex       = rets - rf_d
sharpe   = ex.mean() / rets.std() * np.sqrt(252)
neg_ex   = rets[rets < rf_d] - rf_d
sortino  = ex.mean() / neg_ex.std() * np.sqrt(252) if len(neg_ex) else np.nan
roll_max = pv.cummax()
dd       = (pv - roll_max) / roll_max
max_dd   = dd.min()
in_dd, t0, durs = False, None, []
for d, v in dd.items():
    if v < 0 and not in_dd: in_dd, t0 = True, d
    elif v >= 0 and in_dd:  durs.append((d - t0).days); in_dd = False
if in_dd and t0: durs.append((dd.index[-1] - t0).days)
calmar    = cagr / abs(max_dd) if max_dd else np.nan
spy_cagr  = (spy_n.iloc[-1] / STARTING_CAPITAL) ** (1 / n_yrs) - 1
cm        = rets.index.intersection(spy_r.index)
beta, a, *_ = stats.linregress(spy_r.loc[cm], rets.loc[cm])
alpha_ann = a * 252

longs  = trade_df[trade_df["direction"] ==  1]
shorts = trade_df[trade_df["direction"] == -1]
win_r  = (trade_df["net_return"] > 0).mean() if len(trade_df) else np.nan
l_wr   = (longs["net_return"]  > 0).mean() if len(longs)  else np.nan
s_wr   = (shorts["net_return"] > 0).mean() if len(shorts) else np.nan
avg_r  = trade_df["net_return"].mean() if len(trade_df) else np.nan

# ── Print ─────────────────────────────────────────────────────
W = 30
print("=" * 55)
print(f"  BACKTEST RESULTS  (hold={HOLD_DAYS}d  size={POS_SIZE*100:.0f}%  "
      f"stop={STOP_LOSS if STOP_LOSS else 'off'})")
print("=" * 55)
print(f"  {'CAGR':<{W}} {cagr*100:>+.2f}%   SPY {spy_cagr*100:>+.2f}%")
print(f"  {'Sharpe  (rf={:.0f}%)':<{W}} {sharpe:>+.3f}".format(RF_ANNUAL*100))
print(f"  {'Sortino (rf={:.0f}%)':<{W}} {sortino:>+.3f}".format(RF_ANNUAL*100))
print(f"  {'Calmar':<{W}} {calmar:>+.3f}")
print(f"  {'Max Drawdown':<{W}} {max_dd*100:>+.2f}%")
print(f"  {'Max DD Duration':<{W}} {max(durs) if durs else 0:>} days")
print(f"  {'Total Trades':<{W}} {len(trade_df):>}")
if STOP_LOSS:
    print(f"  {'Stop triggers':<{W}} {int(trade_df['stopped'].sum()):>}")
print(f"  {'Win Rate (all)':<{W}} {win_r*100:>.1f}%")
if not LONG_ONLY:
    print(f"  {'Win Rate (long)':<{W}} {l_wr*100:>.1f}%")
    print(f"  {'Win Rate (short)':<{W}} {s_wr*100:>.1f}%")
print(f"  {'Avg Net Return / Trade':<{W}} {avg_r*100:>+.3f}%")
print(f"  {'Beta vs SPY':<{W}} {beta:>+.3f}")
print(f"  {'Alpha vs SPY (ann.)':<{W}} {alpha_ann*100:>+.2f}%")

# ── Plots ─────────────────────────────────────────────────────
os.makedirs("outputs", exist_ok=True)
STRAT_COLOR = "#2196F3"

# 1. Equity curve
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily_df.index, daily_df["portfolio"],
        label=f"Strategy  CAGR {cagr*100:.1f}%  Sharpe {sharpe:.2f}",
        color=STRAT_COLOR, linewidth=1.5)
ax.plot(spy_n.index, spy_n,
        label=f"SPY B&H   CAGR {spy_cagr*100:.1f}%",
        color="#FF9800", linewidth=1.5, linestyle="--")
ax.axhline(STARTING_CAPITAL, color="gray", linewidth=0.7, linestyle=":")
ax.set_ylabel("Portfolio Value ($)")
ax.set_title(
    f"Sentiment Velocity Strategy  hold={HOLD_DAYS}d  size={POS_SIZE*100:.0f}%  "
    f"stop={'off' if not STOP_LOSS else str(int(STOP_LOSS*100))+'%'}\n"
    f"CAGR {cagr*100:.1f}%  Sharpe {sharpe:.2f}  Sortino {sortino:.2f}  Max DD {max_dd*100:.1f}%"
)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/backtest_equity.png", dpi=150)
plt.close()
print("\nSaved outputs/backtest_equity.png")

# 2. Drawdown
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(dd.index, dd * 100, 0, alpha=0.45, color=STRAT_COLOR,
                label=f"Max DD {max_dd*100:.1f}%  Duration {max(durs) if durs else 0}d")
ax.plot(dd.index, dd * 100, color=STRAT_COLOR, linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylabel("Drawdown (%)")
ax.set_title("Strategy Drawdown")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/backtest_drawdown.png", dpi=150)
plt.close()
print("Saved outputs/backtest_drawdown.png")

# 3. Trade return distribution
fig, ax = plt.subplots(figsize=(10, 5))
if not LONG_ONLY and len(shorts):
    bins = np.linspace(trade_df["net_return"].quantile(0.02)*100,
                       trade_df["net_return"].quantile(0.98)*100, 40)
    ax.hist(longs["net_return"]*100,  bins=bins, alpha=0.65, color="#2196F3",
            label=f"Long  n={len(longs)}  mean={longs['net_return'].mean()*100:+.2f}%")
    ax.hist(shorts["net_return"]*100, bins=bins, alpha=0.65, color="#F44336",
            label=f"Short n={len(shorts)}  mean={shorts['net_return'].mean()*100:+.2f}%")
else:
    bins = np.linspace(trade_df["net_return"].quantile(0.02)*100,
                       trade_df["net_return"].quantile(0.98)*100, 40)
    ax.hist(trade_df["net_return"]*100, bins=bins, alpha=0.75, color=STRAT_COLOR,
            label=f"n={len(trade_df)}  mean={avg_r*100:+.2f}%")
ax.axvline(0,     color="black",      linewidth=1.1, linestyle="--")
ax.axvline(avg_r*100, color=STRAT_COLOR, linewidth=1.8, linestyle="-.",
           label=f"Mean {avg_r*100:+.2f}%")
ax.set_xlabel("Net Return per Trade (%)")
ax.set_ylabel("Count")
ax.set_title(f"Trade Return Distribution  Win Rate {win_r*100:.0f}%")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/backtest_trades.png", dpi=150)
plt.close()
print("Saved outputs/backtest_trades.png")


Signals after filter: 255  (long only  min_vel>0.0)
1959 trading days

Tradeable events: 255  (long=255  short=0)
  BACKTEST RESULTS  (hold=60d  size=15%  stop=off)
  CAGR                           +20.33%   SPY +14.02%
  Sharpe  (rf=4%)           +0.824
  Sortino (rf=4%)           +1.052
  Calmar                         +0.720
  Max Drawdown                   -28.23%
  Max DD Duration                567 days
  Total Trades                   187
  Win Rate (all)                 63.1%
  Avg Net Return / Trade         +6.602%
  Beta vs SPY                    +0.869
  Alpha vs SPY (ann.)            +7.51%

Saved outputs/backtest_equity.png
Saved outputs/backtest_drawdown.png
Saved outputs/backtest_trades.png
